In [1]:
import string
from copy import deepcopy

import sympy as sp
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

In [2]:
# =========================
# ESTADO GLOBAL
# =========================

REGISTRO_MATRICES = {}
HISTORIAL = []

SIMBOLOS_BASE = sp.symbols('a b c d e f g h i j k m n p q r s t x y z')
DICCIONARIO_SIMBOLOS = {str(s): s for s in SIMBOLOS_BASE}

In [3]:
# =========================
# UTILIDADES SIMBÓLICAS
# =========================

def parsear_expresion(expr):
    if isinstance(expr, sp.Basic):
        return sp.simplify(expr)

    texto = str(expr).strip()
    if texto == "":
        raise ValueError("Hay una celda vacía.")

    try:
        return sp.simplify(sp.sympify(texto, locals=DICCIONARIO_SIMBOLOS))
    except Exception as e:
        raise ValueError(f"No se pudo interpretar la expresión: {texto}") from e

def matriz_desde_lista(lista_2d):
    return [[parsear_expresion(v) for v in fila] for fila in lista_2d]

def matriz_a_sympy(M):
    return sp.Matrix(M)

def sympy_a_lista(M):
    return [[sp.simplify(M[i, j]) for j in range(M.cols)] for i in range(M.rows)]

def copiar_matriz(M):
    return deepcopy(M)

def dimensiones(M):
    return len(M), len(M[0])

def es_matriz_valida(M):
    if not isinstance(M, list) or len(M) == 0:
        return False
    if not all(isinstance(fila, list) for fila in M):
        return False
    ncols = len(M[0])
    return all(len(fila) == ncols for fila in M)

def es_cuadrada(M):
    f, c = dimensiones(M)
    return f == c

def matriz_nula(filas, columnas):
    return [[sp.Integer(0) for _ in range(columnas)] for _ in range(filas)]

def identidad(n):
    return sympy_a_lista(sp.eye(n))

def valor_a_texto(x):
    return sp.sstr(sp.simplify(x))

def matriz_a_texto(M, nombre="A"):
    lineas = [f"{nombre} ="]
    for fila in M:
        lineas.append("[ " + "   ".join(valor_a_texto(v) for v in fila) + " ]")
    return "\n".join(lineas)

In [4]:
# =========================
# NOMBRES
# =========================

def siguiente_nombre_base():
    existentes = set(REGISTRO_MATRICES.keys())
    letras = list(string.ascii_uppercase)

    for letra in letras:
        if letra not in existentes:
            return letra

    bloque = 1
    while True:
        for letra in letras:
            candidato = f"{letra}{bloque}"
            if candidato not in existentes:
                return candidato
        bloque += 1

def nombre_transpuesta(nombre):
    return f"{nombre}T"

def nombre_potencia(nombre, exponente):
    sup = {
        "0": "⁰", "1": "¹", "2": "²", "3": "³", "4": "⁴",
        "5": "⁵", "6": "⁶", "7": "⁷", "8": "⁸", "9": "⁹",
        "-": "⁻"
    }
    exp_txt = "".join(sup.get(ch, ch) for ch in str(exponente))
    return f"{nombre}{exp_txt}"

def nombre_escalar(nombre, escalar):
    return f"{valor_a_texto(escalar)}{nombre}"

def nombre_inversa(nombre):
    return f"{nombre}⁻¹"

def nombre_suma(a, b):
    return f"{a}+{b}"

def nombre_resta(a, b):
    return f"{a}-{b}"

def nombre_producto(a, b):
    return f"{a}{b}"

In [5]:
# =========================
# REGISTRO CENTRAL
# =========================

def registrar_matriz(nombre, matriz, origen, procedimiento, sobrescribir=True,
                     procedimiento_html=None, resumen=None, tipo="general"):
    if not es_matriz_valida(matriz):
        raise ValueError("La matriz no tiene formato válido.")

    if (nombre in REGISTRO_MATRICES) and not sobrescribir:
        raise ValueError(f"Ya existe una matriz llamada '{nombre}'.")

    REGISTRO_MATRICES[nombre] = {
        "matriz": copiar_matriz(matriz),
        "dimensiones": dimensiones(matriz),
        "origen": origen,
        "procedimiento": procedimiento,
        "procedimiento_html": procedimiento_html,
        "resumen": resumen,
        "tipo": tipo
    }

    HISTORIAL.append({
        "nombre": nombre,
        "origen": origen,
        "procedimiento": procedimiento,
        "resumen": resumen,
        "tipo": tipo
    })

def obtener_matriz(nombre):
    if nombre not in REGISTRO_MATRICES:
        raise KeyError(f"No existe la matriz '{nombre}'.")
    return copiar_matriz(REGISTRO_MATRICES[nombre]["matriz"])

def obtener_info(nombre):
    if nombre not in REGISTRO_MATRICES:
        raise KeyError(f"No existe la matriz '{nombre}'.")
    return REGISTRO_MATRICES[nombre]

def nombres_matrices():
    return list(REGISTRO_MATRICES.keys())

def eliminar_matriz(nombre):
    if nombre in REGISTRO_MATRICES:
        del REGISTRO_MATRICES[nombre]

def reiniciar_todo():
    global REGISTRO_MATRICES, HISTORIAL
    REGISTRO_MATRICES = {}
    HISTORIAL = []


In [6]:
# =========================
# CREACIÓN DE MATRICES
# =========================

def crear_matriz_manual(lista_2d):
    nombre = siguiente_nombre_base()
    M = matriz_desde_lista(lista_2d)

    procedimiento = (
        f"Se creó manualmente la matriz {nombre} ingresando sus elementos.\n\n"
        f"{matriz_a_texto(M, nombre)}"
    )

    registrar_matriz(
        nombre=nombre,
        matriz=M,
        origen="Creada manualmente",
        procedimiento=procedimiento
    )
    return nombre

def crear_matriz_por_formula(filas, columnas, formula):
    i, j = sp.symbols('i j')
    expr = parsear_expresion(formula)

    M = []
    for fi in range(1, filas + 1):
        fila = []
        for cj in range(1, columnas + 1):
            val = sp.simplify(expr.subs({i: fi, j: cj}))
            fila.append(val)
        M.append(fila)

    nombre = siguiente_nombre_base()

    procedimiento = (
        f"Se creó la matriz {nombre} usando la fórmula:\n"
        f"a(i,j) = {sp.sstr(expr)}\n\n"
        f"con i = 1..{filas} y j = 1..{columnas}\n\n"
        f"{matriz_a_texto(M, nombre)}"
    )

    registrar_matriz(
        nombre=nombre,
        matriz=M,
        origen=f"Creada por fórmula {sp.sstr(expr)}",
        procedimiento=procedimiento
    )
    return nombre

In [7]:
# =========================
# VISUALIZACIÓN
# =========================

def matriz_a_html(nombre, M, origen=None, dimensiones_txt=None):
    filas_html = ""
    for fila in M:
        celdas = "".join(
            f"<td style='padding:6px 12px; text-align:center; font-family:monospace;'>{valor_a_texto(v)}</td>"
            for v in fila
        )
        filas_html += f"<tr>{celdas}</tr>"

    meta = ""
    if dimensiones_txt:
        meta += f"<div style='font-size:12px; color:#555;'>Dimensiones: {dimensiones_txt}</div>"
    if origen:
        meta += f"<div style='font-size:12px; color:#555;'>Origen: {origen}</div>"

    return f"""
    <div style="
        display:inline-block;
        vertical-align:top;
        margin:8px;
        padding:12px;
        border:1px solid #d0d7de;
        border-radius:10px;
        background:white;
        box-shadow:0 1px 3px rgba(0,0,0,0.06);
        min-width:160px;
    ">
        <div style="font-weight:700; text-align:center; margin-bottom:8px; font-size:16px;">
            {nombre}
        </div>

        <div style="display:flex; justify-content:center; align-items:center;">
            <div style="font-size:34px; line-height:1; padding-right:6px;">[</div>
            <table style="border-collapse:collapse;">
                {filas_html}
            </table>
            <div style="font-size:34px; line-height:1; padding-left:6px;">]</div>
        </div>

        <div style="margin-top:8px;">
            {meta}
        </div>
    </div>
    """

def renderizar_panel_matrices():
    if not REGISTRO_MATRICES:
        return HTML("<div style='color:#666; padding:8px;'>No hay matrices guardadas.</div>")

    bloques = []
    for nombre, info in REGISTRO_MATRICES.items():
        f, c = info["dimensiones"]
        bloques.append(
            matriz_a_html(
                nombre=nombre,
                M=info["matriz"],
                origen=info["origen"],
                dimensiones_txt=f"{f}x{c}"
            )
        )

    return HTML(f"""
    <div style="display:flex; flex-wrap:wrap; align-items:flex-start;">
        {''.join(bloques)}
    </div>
    """)

In [8]:
# =========================
# OPERACIONES SIMBÓLICAS
# =========================

def encabezado_metodo(nombre_metodo, formula, explicacion):
    return (
        f"Método usado: {nombre_metodo}.\n\n"
        f"Fórmula general:\n{formula}\n\n"
        f"Aplicación al caso:\n{explicacion}\n\n"
    )

def sumar_matrices(A, B):
    fA, cA = dimensiones(A)
    fB, cB = dimensiones(B)

    if (fA, cA) != (fB, cB):
        raise ValueError("La suma solo está definida para matrices de igual dimensión.")

    R = []
    pasos = []

    for i in range(fA):
        fila = []
        for j in range(cA):
            val = sp.simplify(A[i][j] + B[i][j])
            fila.append(val)
            pasos.append(
                f"({i+1},{j+1}) = {valor_a_texto(A[i][j])} + {valor_a_texto(B[i][j])} = {valor_a_texto(val)}"
            )
        R.append(fila)

    procedimiento = (
        encabezado_metodo(
            "suma de matrices entrada a entrada",
            "(A + B)_(i,j) = a_(i,j) + b_(i,j)",
            "Se suman los elementos que ocupan la misma posición en ambas matrices."
        )
        + "\n".join(pasos)
    )
    return R, procedimiento


def restar_matrices(A, B):
    fA, cA = dimensiones(A)
    fB, cB = dimensiones(B)

    if (fA, cA) != (fB, cB):
        raise ValueError("La resta solo está definida para matrices de igual dimensión.")

    R = []
    pasos = []

    for i in range(fA):
        fila = []
        for j in range(cA):
            val = sp.simplify(A[i][j] - B[i][j])
            fila.append(val)
            pasos.append(
                f"({i+1},{j+1}) = {valor_a_texto(A[i][j])} - {valor_a_texto(B[i][j])} = {valor_a_texto(val)}"
            )
        R.append(fila)

    procedimiento = (
        encabezado_metodo(
            "resta de matrices entrada a entrada",
            "(A - B)_(i,j) = a_(i,j) - b_(i,j)",
            "Se resta en cada posición el elemento de B al correspondiente de A."
        )
        + "\n".join(pasos)
    )
    return R, procedimiento


def multiplicar_por_escalar(k, A):
    k = parsear_expresion(k)

    f, c = dimensiones(A)
    R = []
    pasos = []

    for i in range(f):
        fila = []
        for j in range(c):
            val = sp.simplify(k * A[i][j])
            fila.append(val)
            pasos.append(
                f"({i+1},{j+1}) = {valor_a_texto(k)}·{valor_a_texto(A[i][j])} = {valor_a_texto(val)}"
            )
        R.append(fila)

    procedimiento = (
        encabezado_metodo(
            "producto de un escalar por una matriz",
            "(kA)_(i,j) = k · a_(i,j)",
            f"Cada entrada de la matriz se multiplica por el escalar {valor_a_texto(k)}."
        )
        + "\n".join(pasos)
    )
    return R, procedimiento


def transpuesta_matriz(A):
    f, c = dimensiones(A)
    R = [[sp.simplify(A[i][j]) for i in range(f)] for j in range(c)]

    pasos = []
    for i in range(f):
        for j in range(c):
            pasos.append(
                f"El elemento a({i+1},{j+1}) = {valor_a_texto(A[i][j])} pasa a la posición ({j+1},{i+1})."
            )

    procedimiento = (
        encabezado_metodo(
            "transposición de matrices",
            "(A^T)_(i,j) = a_(j,i)",
            "Las filas de la matriz original pasan a ser las columnas de la transpuesta."
        )
        + "\n".join(pasos)
    )
    return R, procedimiento


def multiplicar_matrices(A, B):
    fA, cA = dimensiones(A)
    fB, cB = dimensiones(B)

    if cA != fB:
        raise ValueError("El producto no está definido: columnas de la primera ≠ filas de la segunda.")

    R = matriz_nula(fA, cB)
    pasos = []

    for i in range(fA):
        for j in range(cB):
            terminos = []
            suma = sp.Integer(0)

            for k in range(cA):
                prod = sp.simplify(A[i][k] * B[k][j])
                suma = sp.simplify(suma + prod)
                terminos.append(f"{valor_a_texto(A[i][k])}·{valor_a_texto(B[k][j])}")

            R[i][j] = sp.simplify(suma)
            pasos.append(
                f"({i+1},{j+1}) = " + " + ".join(terminos) + f" = {valor_a_texto(R[i][j])}"
            )

    procedimiento = (
        encabezado_metodo(
            "producto matricial fila por columna",
            "c_(i,j) = a_(i,1)b_(1,j) + a_(i,2)b_(2,j) + ... + a_(i,n)b_(n,j)",
            f"Cada entrada del resultado se obtiene multiplicando la fila i de la primera matriz por la columna j de la segunda.\n\nCompatibilidad de dimensiones: {fA}x{cA} · {fB}x{cB}"
        )
        + "\n".join(pasos)
    )
    return R, procedimiento


def potencia_matriz(A, n):
    n = int(n)

    if not es_cuadrada(A):
        raise ValueError("La potencia solo está definida para matrices cuadradas.")
    if n < 1:
        raise ValueError("El exponente debe ser mayor o igual a 1.")

    if n == 1:
        return copiar_matriz(A), (
            encabezado_metodo(
                "potencia de matriz",
                "A^1 = A",
                "Toda matriz elevada a la potencia 1 es la misma matriz."
            )
            + "Resultado: A^1 = A"
        )

    resultado = copiar_matriz(A)
    pasos = [f"Se parte de:\n{matriz_a_texto(A, 'A')}"]

    for p in range(2, n + 1):
        resultado, proc = multiplicar_matrices(resultado, A)
        pasos.append(f"\nPaso para obtener A^{p}:\n{proc}")

    procedimiento = (
        encabezado_metodo(
            "potencia de matriz por multiplicación sucesiva",
            "A^n = A · A · ... · A  (n veces)",
            f"Se calcula A^{n} multiplicando sucesivamente la matriz por sí misma."
        )
        + "\n".join(pasos)
    )
    return resultado, procedimiento


In [ ]:
def formatear_matriz_simple(M):
    """Devuelve una matriz en texto plano, útil para mostrar menores en procedimientos."""
    filas = []
    for fila in M:
        filas.append("[ " + "  ".join(valor_a_texto(v) for v in fila) + " ]")
    return "\n".join(filas)


In [9]:
# =========================
# DETERMINANTE CON PROCEDIMIENTO
# =========================

def submatriz_menor(A, fila_excluir, col_excluir):
    return [
        [A[i][j] for j in range(len(A)) if j != col_excluir]
        for i in range(len(A)) if i != fila_excluir
    ]


def determinante_1x1(A):
    det = sp.simplify(A[0][0])
    procedimiento = f"""Método usado: determinante de matriz 1x1.

Fórmula general:
Si A = [a], entonces det(A) = a

Aplicación al caso:
det(A) = {valor_a_texto(det)}"""
    return det, procedimiento


def determinante_2x2(A):
    a, b = A[0]
    c, d = A[1]

    det = sp.simplify(a*d - b*c)

    procedimiento = f"""Método usado: determinante de matriz 2x2.

Fórmula general:
|a  b|
|c  d| = ad - bc

Aplicación al caso:
det(A) = ({valor_a_texto(a)}·{valor_a_texto(d)}) - ({valor_a_texto(b)}·{valor_a_texto(c)})
det(A) = {valor_a_texto(a*d)} - {valor_a_texto(b*c)}
det(A) = {valor_a_texto(det)}"""
    return det, procedimiento


def determinante_3x3_sarrus(A):
    a, b, c = A[0]
    d, e, f = A[1]
    g, h, i = A[2]

    diag1 = sp.simplify(a*e*i)
    diag2 = sp.simplify(b*f*g)
    diag3 = sp.simplify(c*d*h)

    diag4 = sp.simplify(c*e*g)
    diag5 = sp.simplify(a*f*h)
    diag6 = sp.simplify(b*d*i)

    suma_principales = sp.simplify(diag1 + diag2 + diag3)
    suma_secundarias = sp.simplify(diag4 + diag5 + diag6)
    resta_con_parentesis = sp.simplify((diag1 + diag2 + diag3) - (diag4 + diag5 + diag6))
    det = sp.simplify(resta_con_parentesis)

    matriz_repetida = "\n".join([
        f"{valor_a_texto(a)}   {valor_a_texto(b)}   {valor_a_texto(c)}   {valor_a_texto(a)}   {valor_a_texto(b)}",
        f"{valor_a_texto(d)}   {valor_a_texto(e)}   {valor_a_texto(f)}   {valor_a_texto(d)}   {valor_a_texto(e)}",
        f"{valor_a_texto(g)}   {valor_a_texto(h)}   {valor_a_texto(i)}   {valor_a_texto(g)}   {valor_a_texto(h)}",
    ])

    expr_principales = f"({valor_a_texto(a)})({valor_a_texto(e)})({valor_a_texto(i)}) + ({valor_a_texto(b)})({valor_a_texto(f)})({valor_a_texto(g)}) + ({valor_a_texto(c)})({valor_a_texto(d)})({valor_a_texto(h)})"
    expr_secundarias = f"({valor_a_texto(c)})({valor_a_texto(e)})({valor_a_texto(g)}) + ({valor_a_texto(a)})({valor_a_texto(f)})({valor_a_texto(h)}) + ({valor_a_texto(b)})({valor_a_texto(d)})({valor_a_texto(i)})"

    procedimiento = f"""Método usado: regla de Sarrus para determinantes 3x3.

Fórmula general:
det(A) = (suma de diagonales principales) - (suma de diagonales secundarias)

Aplicación al caso:
Paso 1: se repiten las dos primeras columnas a la derecha:

{matriz_repetida}

Paso 2: se suman los productos de las diagonales principales:
{expr_principales}
= {valor_a_texto(diag1)} + {valor_a_texto(diag2)} + {valor_a_texto(diag3)}
= {valor_a_texto(suma_principales)}

Paso 3: se suman los productos de las diagonales secundarias:
{expr_secundarias}
= {valor_a_texto(diag4)} + {valor_a_texto(diag5)} + {valor_a_texto(diag6)}
= {valor_a_texto(suma_secundarias)}

Paso 4: se resta toda la suma de diagonales secundarias entre paréntesis:
det(A) = ({valor_a_texto(suma_principales)}) - ({valor_a_texto(suma_secundarias)})
= ({valor_a_texto(diag1)} + {valor_a_texto(diag2)} + {valor_a_texto(diag3)}) - ({valor_a_texto(diag4)} + {valor_a_texto(diag5)} + {valor_a_texto(diag6)})
= {valor_a_texto(resta_con_parentesis)}
= {valor_a_texto(det)}"""
    return det, procedimiento


def fila_expansion_preferida(A):
    mejor_fila = 0
    mejor_puntaje = None
    for i, fila in enumerate(A):
        no_ceros = [sp.simplify(x) for x in fila if sp.simplify(x) != 0]
        cantidad_no_ceros = len(no_ceros)
        complejidad = sum(len(valor_a_texto(x)) for x in no_ceros)
        puntaje = (cantidad_no_ceros, complejidad, i)
        if mejor_puntaje is None or puntaje < mejor_puntaje:
            mejor_puntaje = puntaje
            mejor_fila = i
    return mejor_fila


def determinante_por_cofactores(A, nivel=0):
    n = len(A)
    pref = "  " * nivel

    if n == 1:
        det, proc = determinante_1x1(A)
        return det, pref + proc.replace("\n", "\n" + pref)

    if n == 2:
        det, proc = determinante_2x2(A)
        return det, pref + proc.replace("\n", "\n" + pref)

    fila = fila_expansion_preferida(A)
    piezas = []
    total = sp.Integer(0)

    piezas.append(pref + f"Expansión por cofactores usando la fila {fila + 1}.")

    for j in range(n):
        a_ij = sp.simplify(A[fila][j])
        if a_ij == 0:
            piezas.append(pref + f"a({fila+1},{j+1}) = 0, este término no aporta.")
            continue

        signo = sp.Integer(1) if (fila + j) % 2 == 0 else sp.Integer(-1)
        menor = submatriz_menor(A, fila, j)
        det_menor, proc_menor = determinante_por_cofactores(menor, nivel + 1)
        cofactor = sp.simplify(signo * det_menor)
        termino = sp.simplify(a_ij * cofactor)
        total = sp.simplify(total + termino)

        piezas.append(
            pref
            + f"Término en columna {j + 1}: a({fila+1},{j+1}) = {valor_a_texto(a_ij)}, signo = {valor_a_texto(signo)}"
        )
        piezas.append(pref + "Menor asociado:")
        piezas.append(pref + formatear_matriz_simple(menor))
        piezas.append(pref + "Determinante del menor:")
        piezas.append(proc_menor)
        piezas.append(
            pref
            + f"Cofactor C({fila+1},{j+1}) = {valor_a_texto(signo)}·{valor_a_texto(det_menor)} = {valor_a_texto(cofactor)}"
        )
        piezas.append(
            pref
            + f"Aporte al determinante: {valor_a_texto(a_ij)}·{valor_a_texto(cofactor)} = {valor_a_texto(termino)}"
        )

    piezas.append(pref + f"Determinante total = {valor_a_texto(total)}")
    return total, "\n".join(piezas)


def datos_determinante_cofactores(A):
    n = len(A)
    if n == 1:
        det, proc = determinante_1x1(A)
        return {
            "metodo": "1x1",
            "det": det,
            "procedimiento": proc,
            "matriz": A,
            "fila": None,
            "terminos": [],
        }

    if n == 2:
        det, proc = determinante_2x2(A)
        return {
            "metodo": "2x2",
            "det": det,
            "procedimiento": proc,
            "matriz": A,
            "fila": None,
            "terminos": [],
        }

    fila = fila_expansion_preferida(A)
    terminos = []
    total = sp.Integer(0)

    for j in range(n):
        a_ij = sp.simplify(A[fila][j])
        if a_ij == 0:
            terminos.append({
                "columna": j,
                "valor": a_ij,
                "signo": None,
                "menor": None,
                "det_menor": None,
                "cofactor": None,
                "aporte": sp.Integer(0),
                "anulado": True,
            })
            continue

        signo = sp.Integer(1) if (fila + j) % 2 == 0 else sp.Integer(-1)
        menor = submatriz_menor(A, fila, j)
        det_menor, _ = determinante_por_cofactores(menor, 0)
        cofactor = sp.simplify(signo * det_menor)
        aporte = sp.simplify(a_ij * cofactor)
        total = sp.simplify(total + aporte)

        terminos.append({
            "columna": j,
            "valor": a_ij,
            "signo": signo,
            "menor": menor,
            "det_menor": det_menor,
            "cofactor": cofactor,
            "aporte": aporte,
            "anulado": False,
        })

    columnas = []
    aportes_texto = []
    for j in range(n):
        a_ij = sp.simplify(A[fila][j])
        columnas.append(f"a({fila + 1},{j + 1})·C({fila + 1},{j + 1})")
        if a_ij == 0:
            aportes_texto.append(f"{valor_a_texto(a_ij)}·C({fila + 1},{j + 1}) = 0")
        else:
            signo = '+' if (fila + j) % 2 == 0 else '-'
            aportes_texto.append(f"{valor_a_texto(a_ij)}·C({fila + 1},{j + 1})")

    expansion_general = " + ".join(columnas)
    expansion_caso = " + ".join(aportes_texto)

    proc = f"""Método usado: expansión por cofactores.

Fórmula general del cofactor:
C(i,j) = (-1)^(i+j) · det(M(i,j))

Fórmula general del determinante por la fila elegida:
det(A) = {expansion_general}

Aplicación al caso:
Se elige la fila {fila + 1} por ser conveniente para expandir.
Entonces:
det(A) = {expansion_caso}

Cada término se obtiene con:
a({fila + 1},j) · C({fila + 1},j)

Determinante total = {valor_a_texto(total)}"""

    return {
        "metodo": "cofactores",
        "det": total,
        "procedimiento": proc,
        "matriz": A,
        "fila": fila,
        "terminos": terminos,
    }


def determinante_matriz(A):
    if not es_cuadrada(A):
        raise ValueError("El determinante solo está definido para matrices cuadradas.")

    n = len(A)
    if n == 1:
        return determinante_1x1(A)
    if n == 2:
        return determinante_2x2(A)
    if n == 3:
        return determinante_3x3_sarrus(A)

    det, proc = determinante_por_cofactores(A, 0)
    encabezado = """Método usado: expansión por cofactores.

Fórmula general:
det(A) = Σ a(i,j) · C(i,j)

Aplicación al caso:
Se elige una fila conveniente y se desarrolla el determinante en términos de menores y cofactores.

"""
    return det, encabezado + proc


In [ ]:
# =========================
# INVERSA Y VERIFICACIONES
# =========================

def tiene_inversa(A):
    if not es_cuadrada(A):
        return False
    det, _ = determinante_matriz(A)
    return sp.simplify(det) != 0


def inversa_matriz_gauss_jordan(A):
    if not es_cuadrada(A):
        raise ValueError("La inversa solo existe para matrices cuadradas.")

    n = len(A)
    A_sym = matriz_a_sympy(A)
    det = sp.simplify(A_sym.det())

    if det == 0:
        raise ValueError("La matriz no tiene inversa porque su determinante es 0.")

    aug = A_sym.row_join(sp.eye(n))
    pasos = []

    pasos.append({
        "titulo": "Matriz aumentada inicial [A | I]",
        "matriz": sympy_a_lista(aug)
    })

    for col in range(n):
        pivote_fila = None
        for fila in range(col, n):
            if sp.simplify(aug[fila, col]) != 0:
                pivote_fila = fila
                break

        if pivote_fila is None:
            raise ValueError("No se encontró pivote; la matriz no es invertible.")

        if pivote_fila != col:
            aug.row_swap(col, pivote_fila)
            pasos.append({
                "titulo": f"Intercambiar F{col+1} ↔ F{pivote_fila+1}",
                "matriz": sympy_a_lista(aug)
            })

        pivote = sp.simplify(aug[col, col])
        if pivote != 1:
            aug.row_op(col, lambda v, j: sp.simplify(v / pivote))
            pasos.append({
                "titulo": f"(1/{valor_a_texto(pivote)})·F{col+1} → F{col+1}",
                "matriz": sympy_a_lista(aug)
            })

        for fila in range(n):
            if fila == col:
                continue

            factor = sp.simplify(aug[fila, col])
            if factor != 0:
                fila_pivote = [aug[col, j] for j in range(2 * n)]
                aug.row_op(fila, lambda v, j: sp.simplify(v - factor * fila_pivote[j]))

                if factor == 1:
                    operacion = f"F{fila+1} - F{col+1} → F{fila+1}"
                elif factor == -1:
                    operacion = f"F{fila+1} + F{col+1} → F{fila+1}"
                else:
                    operacion = f"F{fila+1} - ({valor_a_texto(factor)})F{col+1} → F{fila+1}"

                pasos.append({
                    "titulo": operacion,
                    "matriz": sympy_a_lista(aug)
                })

    inversa = aug[:, n:]
    R = sympy_a_lista(inversa)

    procedimiento_texto = (
        "Método usado: Gauss-Jordan para calcular la inversa.\n\n"
        "Fórmula o idea general:\n"
        "[A | I]  →  [I | A⁻¹]\n\n"
        "Aplicación al caso:\n"
        "Se aplican operaciones elementales por filas hasta convertir la parte izquierda en la identidad. "
        "Cuando eso ocurre, la parte derecha es la inversa de la matriz."
    )

    return R, procedimiento_texto, pasos


def cofactor(A, i, j):
    n = len(A)
    menor = []
    for fila in range(n):
        if fila == i:
            continue
        nueva_fila = []
        for col in range(n):
            if col == j:
                continue
            nueva_fila.append(A[fila][col])
        menor.append(nueva_fila)

    det_menor, _ = determinante_matriz(menor)
    return sp.simplify(((-1) ** (i + j)) * det_menor), menor


def matriz_cofactores(A):
    if not es_cuadrada(A):
        raise ValueError("La matriz de cofactores solo existe para matrices cuadradas.")

    n = len(A)
    C = []
    pasos = []

    for i in range(n):
        fila = []
        for j in range(n):
            c_ij, menor = cofactor(A, i, j)
            fila.append(c_ij)
            pasos.append({
                "i": i,
                "j": j,
                "menor": menor,
                "cofactor": c_ij
            })
        C.append(fila)

    return C, pasos


def inversa_por_cofactores(A):
    if not es_cuadrada(A):
        raise ValueError("La inversa solo existe para matrices cuadradas.")

    detA, proc_det = determinante_matriz(A)
    detA = sp.simplify(detA)

    if detA == 0:
        raise ValueError("La matriz no tiene inversa porque su determinante es 0.")

    C, pasos_cof = matriz_cofactores(A)
    Adj, _ = transpuesta_matriz(C)
    Inv = [[sp.simplify(Adj[i][j] / detA) for j in range(len(Adj[0]))] for i in range(len(Adj))]

    procedimiento = (
        "Método usado: inversa por matriz de cofactores.\n\n"
        "Fórmula general:\n"
        "A⁻¹ = (1/det(A)) · Adj(A), con Adj(A) = (Cof(A))^T\n\n"
        "Aplicación al caso:\n"
        f"Primero se calcula det(A) = {valor_a_texto(detA)}.\n"
        "Luego se construye la matriz de cofactores Cof(A).\n"
        "Después se transpone para obtener la adjunta Adj(A) = (Cof(A))^T.\n"
        "Finalmente, se multiplica la adjunta por 1/det(A)."
    )

    return Inv, procedimiento, {
        "det": detA,
        "proc_det": proc_det,
        "cofactores": C,
        "adjunta": Adj,
        "pasos_cofactores": pasos_cof
    }


def inversa_matriz(A, metodo="gauss"):
    metodo = metodo.lower().strip()

    if metodo == "gauss":
        R, proc_texto, pasos = inversa_matriz_gauss_jordan(A)
        return R, proc_texto, {"metodo": "gauss", "pasos": pasos}

    if metodo == "cofactores":
        R, proc_texto, datos = inversa_por_cofactores(A)
        datos["metodo"] = "cofactores"
        return R, proc_texto, datos

    raise ValueError("Método no válido. Use 'gauss' o 'cofactores'.")


def matrices_iguales(A, B):
    if dimensiones(A) != dimensiones(B):
        return False

    f, c = dimensiones(A)
    for i in range(f):
        for j in range(c):
            if sp.simplify(A[i][j] - B[i][j]) != 0:
                return False

    return True


In [11]:
# =========================
# GUARDADO AUTOMÁTICO
# =========================

def registrar_resultado(nombre, matriz, origen, procedimiento,
                        procedimiento_html=None, resumen=None, tipo="general"):
    registrar_matriz(
        nombre=nombre,
        matriz=matriz,
        origen=origen,
        procedimiento=procedimiento,
        sobrescribir=True,
        procedimiento_html=procedimiento_html,
        resumen=resumen,
        tipo=tipo
    )

def transformar_y_guardar(nombre_base, hacer_transpuesta=False, hacer_escalar=False, escalar=None,
                           hacer_potencia=False, exponente=None, hacer_inversa=False):
    A = obtener_matriz(nombre_base)
    creadas = []

    if hacer_transpuesta:
        R, proc = transpuesta_matriz(A)
        nombre = nombre_transpuesta(nombre_base)
        registrar_resultado(
            nombre, R,
            f"Transpuesta de {nombre_base}",
            proc,
            resumen=f"Se generó la transpuesta <b>{nombre}</b> a partir de <b>{nombre_base}</b>.",
            tipo="transformación"
        )
        creadas.append(nombre)

    if hacer_escalar:
        k = parsear_expresion(escalar)
        R, proc = multiplicar_por_escalar(escalar, A)
        nombre = nombre_escalar(nombre_base, k)
        registrar_resultado(
            nombre, R,
            f"Escalar por {valor_a_texto(k)} de {nombre_base}",
            proc,
            resumen=f"Se generó <b>{nombre}</b> multiplicando <b>{nombre_base}</b> por <b>{valor_a_texto(k)}</b>.",
            tipo="transformación"
        )
        creadas.append(nombre)

    if hacer_potencia:
        R, proc = potencia_matriz(A, exponente)
        nombre = nombre_potencia(nombre_base, exponente)
        registrar_resultado(
            nombre, R,
            f"Potencia {exponente} de {nombre_base}",
            proc,
            resumen=f"Se generó <b>{nombre}</b> elevando <b>{nombre_base}</b> a la potencia <b>{exponente}</b>.",
            tipo="transformación"
        )
        creadas.append(nombre)

    if hacer_inversa:
        inv = inversa_matriz(A, metodo="gauss")
        if isinstance(inv, tuple) and len(inv) == 3:
            R, proc, _ = inv
        else:
            R, proc = inv
        nombre = nombre_inversa(nombre_base)
        registrar_resultado(
            nombre, R,
            f"Inversa de {nombre_base}",
            proc,
            resumen=f"Se generó la inversa <b>{nombre}</b> de la matriz <b>{nombre_base}</b>.",
            tipo="inversa"
        )
        creadas.append(nombre)

    return creadas


def operar_y_guardar(nombre1, operacion, nombre2=None, escalar=None):
    A = obtener_matriz(nombre1)

    if operacion == "Suma":
        B = obtener_matriz(nombre2)
        R, proc = sumar_matrices(A, B)
        nombre = nombre_suma(nombre1, nombre2)
        origen = f"Suma de {nombre1} y {nombre2}"
        resumen = f"Se guardó <b>{nombre}</b> como suma de <b>{nombre1}</b> y <b>{nombre2}</b>."

    elif operacion == "Resta":
        B = obtener_matriz(nombre2)
        R, proc = restar_matrices(A, B)
        nombre = nombre_resta(nombre1, nombre2)
        origen = f"Resta de {nombre1} y {nombre2}"
        resumen = f"Se guardó <b>{nombre}</b> como resta de <b>{nombre1}</b> y <b>{nombre2}</b>."

    elif operacion == "Producto":
        B = obtener_matriz(nombre2)
        R, proc = multiplicar_matrices(A, B)
        nombre = nombre_producto(nombre1, nombre2)
        origen = f"Producto de {nombre1} y {nombre2}"
        resumen = f"Se guardó <b>{nombre}</b> como producto de <b>{nombre1}</b> y <b>{nombre2}</b>."

    elif operacion == "Escalar":
        k = parsear_expresion(escalar)
        R, proc = multiplicar_por_escalar(escalar, A)
        nombre = nombre_escalar(nombre1, k)
        origen = f"Escalar por {valor_a_texto(k)} de {nombre1}"
        resumen = f"Se guardó <b>{nombre}</b> como producto escalar de <b>{nombre1}</b> por <b>{valor_a_texto(k)}</b>."

    else:
        raise ValueError("Operación no reconocida.")

    registrar_resultado(
        nombre, R, origen, proc,
        resumen=resumen,
        tipo="operación"
    )
    return nombre


In [ ]:

# =========================
# RESOLVEDORES DE EJERCICIOS
# =========================
import re

def crear_identidad(n):
    return [[sp.Integer(1) if i == j else sp.Integer(0) for j in range(n)] for i in range(n)]


def crear_nula(filas, columnas):
    return [[sp.Integer(0) for _ in range(columnas)] for _ in range(filas)]


def crear_matriz_por_casos(filas, columnas, condicion, expr_si, expr_no):
    i, j = sp.symbols('i j')
    expr_cond = sp.sympify(condicion)
    expr_si = parsear_expresion(expr_si)
    expr_no = parsear_expresion(expr_no)

    M = []
    for fi in range(1, filas + 1):
        fila = []
        for cj in range(1, columnas + 1):
            cond_val = bool(expr_cond.subs({i: fi, j: cj}))
            expr = expr_si if cond_val else expr_no
            fila.append(sp.simplify(expr.subs({i: fi, j: cj})))
        M.append(fila)

    procedimiento = (
        f"Se creó una matriz por casos con condición: {sp.sstr(expr_cond)}\n"
        f"Si se cumple, a(i,j) = {sp.sstr(expr_si)}\n"
        f"Si no se cumple, a(i,j) = {sp.sstr(expr_no)}\n"
    )
    return M, procedimiento


def _entorno_matricial(datos=None, dimension_referencia=None):
    env = {"sp": sp}
    datos = datos or {}
    for nombre, matriz in datos.items():
        env[nombre] = matriz_a_sympy(matriz)

    if dimension_referencia is None and datos:
        primera = next(iter(datos.values()))
        if es_cuadrada(primera):
            dimension_referencia = len(primera)

    if dimension_referencia is not None:
        env['I'] = sp.eye(dimension_referencia)
        env['O'] = sp.zeros(dimension_referencia, dimension_referencia)

    return env


def evaluar_expresion_matricial(expresion, datos=None, dimension_referencia=None):
    env = _entorno_matricial(datos, dimension_referencia)
    expr = expresion.replace('^', '**')
    try:
        valor = eval(expr, {"__builtins__": {}}, env)
    except Exception as e:
        raise ValueError(f"No se pudo evaluar la expresión matricial '{expresion}': {e}")

    if isinstance(valor, sp.MatrixBase):
        return sympy_a_lista(valor)
    raise ValueError(f"La expresión '{expresion}' no produjo una matriz.")


def evaluar_expresion_escalar(expresion):
    return sp.simplify(parsear_expresion(expresion))


def resolver_AX_B(A, B, metodo='gauss'):
    if not es_cuadrada(A):
        raise ValueError('Para resolver AX = B, la matriz A debe ser cuadrada.')
    if len(A) != len(B):
        raise ValueError('Las dimensiones no son compatibles en AX = B.')

    A_inv, proc_inv, datos_inv = inversa_matriz(A, metodo=metodo)
    X, proc_mult = multiplicar_matrices(A_inv, B)

    procedimiento = (
        'Método usado: despeje matricial multiplicando por la inversa a la izquierda.\n\n'
        'Fórmula general:\n'
        'Si AX = B y A es invertible, entonces X = A⁻¹B\n\n'
        'Aplicación al caso:\n'
        'Partimos de AX = B y multiplicamos ambos lados a izquierda por A⁻¹:\n'
        'A⁻¹AX = A⁻¹B\n'
        'IX = A⁻¹B\n'
        'X = A⁻¹B\n\n'
        + proc_inv + '\n\n'
        + proc_mult
    )
    return X, procedimiento, {
        'metodo': metodo,
        'inversa': A_inv,
        'datos_inversa': datos_inv
    }


def resolver_XA_B(A, B, metodo='gauss'):
    if not es_cuadrada(A):
        raise ValueError('Para resolver XA = B, la matriz A debe ser cuadrada.')
    if len(A[0]) != len(B[0]):
        raise ValueError('Las dimensiones no son compatibles en XA = B.')

    A_inv, proc_inv, datos_inv = inversa_matriz(A, metodo=metodo)
    X, proc_mult = multiplicar_matrices(B, A_inv)

    procedimiento = (
        'Método usado: despeje matricial multiplicando por la inversa a la derecha.\n\n'
        'Fórmula general:\n'
        'Si XA = B y A es invertible, entonces X = BA⁻¹\n\n'
        'Aplicación al caso:\n'
        'Partimos de XA = B y multiplicamos ambos lados a derecha por A⁻¹:\n'
        'XAA⁻¹ = BA⁻¹\n'
        'XI = BA⁻¹\n'
        'X = BA⁻¹\n\n'
        + proc_inv + '\n\n'
        + proc_mult
    )
    return X, procedimiento, {
        'metodo': metodo,
        'inversa': A_inv,
        'datos_inversa': datos_inv
    }


def _split_top_level(expr):
    partes = []
    actual = []
    nivel = 0
    for k, ch in enumerate(expr):
        if ch == '(':
            nivel += 1
        elif ch == ')':
            nivel -= 1
        if nivel == 0 and ch in '+-' and k > 0:
            partes.append(''.join(actual).strip())
            actual = [ch]
        else:
            actual.append(ch)
    if actual:
        partes.append(''.join(actual).strip())
    return [p for p in partes if p]


def _coeficiente_de_X(termino):
    t = termino.replace(' ', '')
    if 'X' not in t:
        raise ValueError('El término no contiene X.')
    if t == 'X' or t == '+X':
        return sp.Integer(1)
    if t == '-X':
        return sp.Integer(-1)
    if t.endswith('*X'):
        return evaluar_expresion_escalar(t[:-2])
    if t.endswith('X'):
        return evaluar_expresion_escalar(t[:-1])
    raise ValueError(
        'Solo se admiten términos lineales en X, como X, -X, 2*X o 2X.'
    )


def _sumar_lista_matrices(lista_mats):
    if not lista_mats:
        raise ValueError('No hay matrices para sumar.')
    acc = copiar_matriz(lista_mats[0])
    for M in lista_mats[1:]:
        acc, _ = sumar_matrices(acc, M)
    return acc


def _analizar_lado_ecuacion(expr, datos):
    terminos = _split_top_level(expr)
    coef_x = sp.Integer(0)
    mats = []
    detalles = []

    for termino in terminos:
        if 'X' in termino.replace(' ', ''):
            coef = _coeficiente_de_X(termino)
            coef_x = sp.simplify(coef_x + coef)
            detalles.append(f"Término con X: {termino}  →  coeficiente {valor_a_texto(coef)}")
        else:
            M = evaluar_expresion_matricial(termino, datos)
            mats.append(M)
            detalles.append(f"Término matricial: {termino}")

    suma_mats = None
    if mats:
        suma_mats = _sumar_lista_matrices(mats)

    return coef_x, suma_mats, detalles


def resolver_ecuacion_matricial_lineal(expr_izq, expr_der, datos):
    coef_izq, mats_izq, det_izq = _analizar_lado_ecuacion(expr_izq, datos)
    coef_der, mats_der, det_der = _analizar_lado_ecuacion(expr_der, datos)

    if mats_izq is None and mats_der is None:
        raise ValueError('La ecuación no contiene parte matricial suficiente.')

    if mats_izq is None:
        mats_izq = crear_nula(*dimensiones(mats_der))
    if mats_der is None:
        mats_der = crear_nula(*dimensiones(mats_izq))

    coef_total = sp.simplify(coef_izq - coef_der)
    if coef_total == 0:
        raise ValueError('El coeficiente total de X es 0; no se puede despejar de forma única.')

    diff, proc_resta = restar_matrices(mats_der, mats_izq)
    X = [[sp.simplify(v / coef_total) for v in fila] for fila in diff]

    procedimiento = (
        f"Ecuación original: {expr_izq} = {expr_der}\n\n"
        + '\n'.join(det_izq + det_der) + '\n\n'
        + f"Coeficiente de X en el lado izquierdo: {valor_a_texto(coef_izq)}\n"
        + f"Coeficiente de X en el lado derecho: {valor_a_texto(coef_der)}\n"
        + f"Coeficiente total al pasar términos: {valor_a_texto(coef_total)}\n\n"
        + 'Se pasa toda la parte matricial sin X al otro lado y se obtiene:\n'
        + f"({valor_a_texto(coef_total)})X = M_derecha - M_izquierda\n\n"
        + proc_resta + '\n\n'
        + f"Ahora dividimos cada entrada entre {valor_a_texto(coef_total)} para obtener X."
    )

    return X, procedimiento, {
        'coef_total': coef_total,
        'matriz_diferencia': diff
    }


def verificar_identidad_matricial(expr_izq, expr_der, datos):
    dim_ref = None
    if datos:
        primera = next(iter(datos.values()))
        if es_cuadrada(primera):
            dim_ref = len(primera)

    A = evaluar_expresion_matricial(expr_izq, datos, dim_ref)
    B = evaluar_expresion_matricial(expr_der, datos, dim_ref)
    iguales = matrices_iguales(A, B)

    diferencia, proc_resta = restar_matrices(A, B)
    procedimiento = (
        f"Se evalúa el lado izquierdo: {expr_izq}\n"
        f"Se evalúa el lado derecho: {expr_der}\n\n"
        + proc_resta + '\n\n'
        + ('Como la diferencia es la matriz nula, la identidad es verdadera.'
           if iguales else 'Como la diferencia no es la matriz nula, la identidad es falsa.')
    )
    return iguales, procedimiento, {
        'izquierda': A,
        'derecha': B,
        'diferencia': diferencia
    }


def resolver_parametro_determinante(expr_matriz, variable='t', valor_objetivo=0, datos=None):
    var = sp.symbols(variable)
    datos = datos or {}

    dim_ref = None
    if datos:
        primera = next(iter(datos.values()))
        if es_cuadrada(primera):
            dim_ref = len(primera)

    env = _entorno_matricial(datos, dim_ref)
    env[variable] = var

    expr = expr_matriz.replace('^', '**')
    try:
        M = eval(expr, {"__builtins__": {}}, env)
    except Exception as e:
        raise ValueError(f"No se pudo evaluar la expresión '{expr_matriz}': {e}")

    if not isinstance(M, sp.MatrixBase):
        raise ValueError('La expresión dada no produce una matriz.')

    det_expr = sp.simplify(M.det())
    objetivo = sp.simplify(valor_objetivo)
    soluciones = sp.solve(sp.Eq(det_expr, objetivo), var)

    procedimiento = (
        f"Se construye la matriz dada por {expr_matriz}.\n"
        f"Luego se calcula su determinante:\n"
        f"det = {sp.sstr(det_expr)}\n\n"
        f"Se resuelve la ecuación det = {sp.sstr(objetivo)} respecto de {variable}.\n"
        f"Soluciones: {soluciones}"
    )

    return soluciones, procedimiento, {
        'determinante': det_expr,
        'variable': var,
        'matriz': sympy_a_lista(M)
    }


def verificar_auto_inversa(A):
    if not es_cuadrada(A):
        raise ValueError('La matriz debe ser cuadrada.')
    A2, proc = multiplicar_matrices(A, A)
    I = crear_identidad(len(A))
    es_auto = matrices_iguales(A2, I)
    procedimiento = proc + '\n\n' + ('Se obtiene la identidad, así que A es su propia inversa.' if es_auto else 'No se obtiene la identidad, así que A no es su propia inversa.')
    return es_auto, procedimiento, {'A2': A2, 'I': I}


def es_matriz_probabilidad(P, tolerar_simbolico=False):
    filas, columnas = dimensiones(P)
    sumas = []
    no_negativos = True

    for fila in P:
        suma = sp.simplify(sum(fila))
        sumas.append(suma)
        for v in fila:
            vv = sp.simplify(v)
            if vv.is_number:
                if vv < 0:
                    no_negativos = False
            elif not tolerar_simbolico:
                raise ValueError('La matriz contiene entradas simbólicas y no se pueden validar como probabilidades numéricas.')

    filas_ok = all(sp.simplify(s - 1) == 0 for s in sumas)
    ok = no_negativos and filas_ok
    procedimiento = (
        'Se verifica que todas las entradas sean no negativas y que la suma de cada fila sea 1.\n\n'
        + '\n'.join(f'Fila {i+1}: suma = {valor_a_texto(s)}' for i, s in enumerate(sumas))
        + '\n\n'
        + ('La matriz sí es de probabilidad.' if ok else 'La matriz no es de probabilidad.')
    )
    return ok, procedimiento, {'sumas_fila': sumas, 'no_negativos': no_negativos}


def verificar_probabilidad_producto(P, Q):
    PQ, proc_prod = multiplicar_matrices(P, Q)
    ok, proc_prob, datos = es_matriz_probabilidad(PQ)
    procedimiento = proc_prod + '\n\n' + proc_prob
    return ok, procedimiento, {'producto': PQ, **datos}


def resolver_coeficientes_polinomio_matriz(A, variables=('b', 'c')):
    if not es_cuadrada(A):
        raise ValueError('La matriz debe ser cuadrada.')
    if len(variables) != 2:
        raise ValueError('Se esperan exactamente dos variables, por ejemplo b y c.')

    b, c = sp.symbols(' '.join(variables))
    A_sym = matriz_a_sympy(A)
    I = sp.eye(len(A))
    expr = A_sym**2 + b*A_sym + c*I
    ecuaciones = [sp.Eq(sp.simplify(expr[i, j]), 0) for i in range(expr.rows) for j in range(expr.cols)]
    soluciones = sp.solve(ecuaciones, (b, c), dict=True)

    procedimiento = (
        f'Se plantea la ecuación matricial A^2 + {variables[0]}A + {variables[1]}I = O.\n\n'
        f'A^2 + {variables[0]}A + {variables[1]}I =\n{sp.sstr(expr)}\n\n'
        'Luego se iguala cada entrada a 0 y se resuelve el sistema resultante.\n'
        f'Soluciones: {soluciones}'
    )

    return soluciones, procedimiento, {
        'expresion': sympy_a_lista(expr),
        'variables': variables
    }


def resolver_variables_escalaras_en_igualdad(expr_izq, expr_der, variables, datos=None):
    datos = datos or {}
    nombres_vars = [v.strip() for v in variables if v.strip()]
    if not nombres_vars:
        raise ValueError('Debes indicar al menos una variable escalar.')

    simbolos = sp.symbols(' '.join(nombres_vars))
    if len(nombres_vars) == 1:
        simbolos = (simbolos,)

    env = _entorno_matricial(datos)
    env.update({nombre: simbolo for nombre, simbolo in zip(nombres_vars, simbolos)})
    env['Matrix'] = sp.Matrix
    env['col'] = lambda *args: sp.Matrix(list(args))
    env['row'] = lambda *args: sp.Matrix([list(args)])

    izquierda = eval(expr_izq.replace('^', '**'), {"__builtins__": {}}, env)
    derecha = eval(expr_der.replace('^', '**'), {"__builtins__": {}}, env)

    if not isinstance(izquierda, sp.MatrixBase) or not isinstance(derecha, sp.MatrixBase):
        raise ValueError('Ambas expresiones deben producir matrices o vectores columna/fila.')
    if izquierda.shape != derecha.shape:
        raise ValueError('Las expresiones evaluadas no tienen las mismas dimensiones.')

    ecuaciones = [
        sp.Eq(sp.simplify(izquierda[i, j] - derecha[i, j]), 0)
        for i in range(izquierda.rows)
        for j in range(izquierda.cols)
    ]
    soluciones = sp.solve(ecuaciones, simbolos, dict=True)

    procedimiento = (
        f'Se evalúan las expresiones\nIzquierda: {expr_izq}\nDerecha: {expr_der}\n\n'
        'Luego se iguala entrada por entrada y se resuelve el sistema escalar resultante.\n'
        f'Soluciones: {soluciones}'
    )

    return soluciones, procedimiento, {
        'izquierda': sympy_a_lista(izquierda),
        'derecha': sympy_a_lista(derecha),
        'variables': nombres_vars
    }


In [ ]:
# Celda reservada: la versión oficial de la inversa ya está definida arriba.


In [12]:
def html_sarrus_3x3(A, nombre="A"):
    a, b, c = A[0]
    d, e, f = A[1]
    g, h, i = A[2]

    diag1 = sp.simplify(a*e*i)
    diag2 = sp.simplify(b*f*g)
    diag3 = sp.simplify(c*d*h)

    diag4 = sp.simplify(c*e*g)
    diag5 = sp.simplify(a*f*h)
    diag6 = sp.simplify(b*d*i)

    suma_principales = sp.simplify(diag1 + diag2 + diag3)
    suma_secundarias = sp.simplify(diag4 + diag5 + diag6)
    resta_con_parentesis = sp.simplify((diag1 + diag2 + diag3) - (diag4 + diag5 + diag6))
    det = sp.simplify(resta_con_parentesis)

    M = [
        [a, b, c, a, b],
        [d, e, f, d, e],
        [g, h, i, g, h],
    ]

    filas_html = ""
    for fila in M:
        celdas = ""
        for idx, v in enumerate(fila):
            borde = "border-right:2px solid #94a3b8;" if idx == 2 else ""
            bg = "#eef6ff" if idx >= 3 else "white"
            celdas += f"""
            <td style="
                padding:8px 12px;
                text-align:center;
                font-family:monospace;
                font-size:16px;
                border:1px solid #cbd5e1;
                {borde}
                background:{bg};
            ">{valor_a_texto(v)}</td>
            """
        filas_html += f"<tr>{celdas}</tr>"

    expr_principales = f"({valor_a_texto(a)})({valor_a_texto(e)})({valor_a_texto(i)}) + ({valor_a_texto(b)})({valor_a_texto(f)})({valor_a_texto(g)}) + ({valor_a_texto(c)})({valor_a_texto(d)})({valor_a_texto(h)})"
    expr_secundarias = f"({valor_a_texto(c)})({valor_a_texto(e)})({valor_a_texto(g)}) + ({valor_a_texto(a)})({valor_a_texto(f)})({valor_a_texto(h)}) + ({valor_a_texto(b)})({valor_a_texto(d)})({valor_a_texto(i)})"

    html = f"""
    <div style="font-family:Arial, sans-serif; line-height:1.5;">

        <div style="font-size:22px; font-weight:700; margin-bottom:10px;">
            Regla de Sarrus
        </div>

        <div style="margin-bottom:10px; background:#f8fafc; border:1px solid #e2e8f0; border-radius:10px; padding:12px;">
            <b>Fórmula usada:</b><br>
            det({nombre}) = (suma de diagonales principales) - (suma de diagonales secundarias)
        </div>

        <div style="margin-bottom:14px;">
            <b>Paso 1.</b> Repetimos las dos primeras columnas de la matriz {nombre}.
        </div>

        <table style="border-collapse:collapse; margin:8px 0 18px 0;">
            {filas_html}
        </table>

        <div style="
            background:#f8fafc;
            border:1px solid #e2e8f0;
            border-radius:10px;
            padding:12px;
            margin:12px 0;
        ">
            <div style="font-weight:700; color:#0f172a; margin-bottom:8px;">
                Paso 2. Diagonales principales
            </div>
            <div style="font-family:monospace; font-size:15px;">
                {expr_principales}
            </div>
            <div style="font-family:monospace; font-size:15px; margin-top:6px;">
                = {valor_a_texto(diag1)} + {valor_a_texto(diag2)} + {valor_a_texto(diag3)}
            </div>
            <div style="font-family:monospace; font-size:16px; font-weight:700; margin-top:6px;">
                = {valor_a_texto(suma_principales)}
            </div>
        </div>

        <div style="
            background:#fff7ed;
            border:1px solid #fed7aa;
            border-radius:10px;
            padding:12px;
            margin:12px 0;
        ">
            <div style="font-weight:700; color:#7c2d12; margin-bottom:8px;">
                Paso 3. Diagonales secundarias
            </div>
            <div style="font-family:monospace; font-size:15px;">
                {expr_secundarias}
            </div>
            <div style="font-family:monospace; font-size:15px; margin-top:6px;">
                = {valor_a_texto(diag4)} + {valor_a_texto(diag5)} + {valor_a_texto(diag6)}
            </div>
            <div style="font-family:monospace; font-size:16px; font-weight:700; margin-top:6px;">
                = {valor_a_texto(suma_secundarias)}
            </div>
        </div>

        <div style="
            background:#ecfdf5;
            border:1px solid #a7f3d0;
            border-radius:10px;
            padding:12px;
            margin-top:14px;
        ">
            <div style="font-weight:700; color:#065f46; margin-bottom:8px;">
                Paso 4. Resta final
            </div>
            <div style="font-family:monospace; font-size:16px; margin-bottom:6px;">
                det({nombre}) = ({valor_a_texto(suma_principales)}) - ({valor_a_texto(suma_secundarias)})
            </div>
            <div style="font-family:monospace; font-size:15px; margin-bottom:6px;">
                = ({valor_a_texto(diag1)} + {valor_a_texto(diag2)} + {valor_a_texto(diag3)}) - ({valor_a_texto(diag4)} + {valor_a_texto(diag5)} + {valor_a_texto(diag6)})
            </div>
            <div style="font-family:monospace; font-size:15px; margin-bottom:6px;">
                = {valor_a_texto(resta_con_parentesis)}
            </div>
            <div style="font-family:monospace; font-size:18px; font-weight:700;">
                = {valor_a_texto(det)}
            </div>
        </div>
    </div>
    """
    return html


In [ ]:

def tarjeta_html(titulo, cuerpo, color="#2563eb", fondo="#eff6ff"):
    return f"""
    <div style="
        border:1px solid #dbeafe;
        border-left:6px solid {color};
        background:{fondo};
        padding:12px 14px;
        border-radius:10px;
        margin:8px 0 12px 0;
    ">
        <div style="font-weight:700; font-size:16px; margin-bottom:6px;">{titulo}</div>
        <div style="line-height:1.45;">{cuerpo}</div>
    </div>
    """

def bloque_texto_pre(texto):
    return f"""
    <pre style="
        background:#f8fafc;
        border:1px solid #e2e8f0;
        border-radius:8px;
        padding:12px;
        white-space:pre-wrap;
        font-family:monospace;
        font-size:14px;
        line-height:1.45;
        margin:8px 0;
    ">{texto}</pre>
    """

def html_error(mensaje):
    return tarjeta_html(
        "Error",
        f"<span style='color:#991b1b;'>{mensaje}</span>",
        color="#dc2626",
        fondo="#fef2f2"
    )

def html_info(mensaje):
    return tarjeta_html(
        "Información",
        mensaje,
        color="#64748b",
        fondo="#f8fafc"
    )

def html_resultado_transformacion(nombres_creados):
    items = "".join(f"<li style='margin:4px 0;'><b>{nombre}</b></li>" for nombre in nombres_creados)
    return tarjeta_html(
        "Transformaciones generadas",
        f"<div>Se crearon las siguientes matrices:</div><ul style='margin:8px 0 0 18px;'>{items}</ul>",
        color="#0891b2",
        fondo="#ecfeff"
    )

def html_resultado_operacion(nombre_res, info):
    f, c = info["dimensiones"]
    resumen = info.get("resumen") or f"Resultado guardado como <b>{nombre_res}</b>."
    proc = info.get("procedimiento", "")
    matriz_html = matriz_a_html(
        nombre=nombre_res,
        M=info["matriz"],
        origen=info["origen"],
        dimensiones_txt=f"{f}x{c}"
    )
    return f"""
    {tarjeta_html("Resultado de la operación", resumen, color="#16a34a", fondo="#f0fdf4")}
    {matriz_html}
    <details style="margin-top:10px;">
        <summary style="cursor:pointer; font-weight:700;">Ver procedimiento completo</summary>
        {bloque_texto_pre(proc)}
    </details>
    """

def html_procedimiento_matriz(nombre, info):
    f, c = info["dimensiones"]
    resumen = info.get("resumen")
    proc_html = info.get("procedimiento_html")
    proc_texto = info.get("procedimiento", "")
    cabecera = tarjeta_html(
        f"Procedimiento de {nombre}",
        f"""
        <div><b>Origen:</b> {info['origen']}</div>
        <div><b>Dimensiones:</b> {f}x{c}</div>
        {f"<div style='margin-top:6px;'><b>Resumen:</b> {resumen}</div>" if resumen else ""}
        """,
        color="#7c3aed",
        fondo="#faf5ff"
    )
    matriz_html = matriz_a_html(
        nombre=nombre,
        M=info["matriz"],
        origen=info["origen"],
        dimensiones_txt=f"{f}x{c}"
    )
    cuerpo = proc_html if proc_html else bloque_texto_pre(proc_texto)
    return f"""{cabecera}{matriz_html}<div style="margin-top:14px;">{cuerpo}</div>"""

def html_historial():
    if not HISTORIAL:
        return html_info("No hay historial todavía.")
    filas = ""
    for i, item in enumerate(HISTORIAL, start=1):
        resumen = item.get("resumen") or item["origen"]
        tipo = item.get("tipo", "general")
        filas += f"""
        <tr>
            <td style="padding:8px 10px; border:1px solid #e5e7eb;">{i}</td>
            <td style="padding:8px 10px; border:1px solid #e5e7eb;"><b>{item['nombre']}</b></td>
            <td style="padding:8px 10px; border:1px solid #e5e7eb;">{tipo}</td>
            <td style="padding:8px 10px; border:1px solid #e5e7eb;">{item['origen']}</td>
            <td style="padding:8px 10px; border:1px solid #e5e7eb;">{resumen}</td>
        </tr>
        """
    return f"""
    <div style="margin-top:8px;">
        <table style="border-collapse:collapse; width:100%; background:white;">
            <thead>
                <tr style="background:#f8fafc;">
                    <th style="padding:8px 10px; border:1px solid #e5e7eb;">#</th>
                    <th style="padding:8px 10px; border:1px solid #e5e7eb;">Nombre</th>
                    <th style="padding:8px 10px; border:1px solid #e5e7eb;">Tipo</th>
                    <th style="padding:8px 10px; border:1px solid #e5e7eb;">Origen</th>
                    <th style="padding:8px 10px; border:1px solid #e5e7eb;">Resumen</th>
                </tr>
            </thead>
            <tbody>{filas}</tbody>
        </table>
    </div>
    """


# =========================
# RENDER HTML PARA MATRIZ AUMENTADA
# =========================

def html_matriz_aumentada(M, n_izq, titulo=None):
    filas = len(M)
    cols = len(M[0])

    filas_html = ""
    for i in range(filas):
        celdas = ""
        for j in range(cols):
            borde = "border-right:3px solid #64748b;" if j == n_izq - 1 else ""
            celdas += f"""
            <td style="
                padding:6px 10px;
                text-align:center;
                font-family:monospace;
                font-size:15px;
                border:1px solid #cbd5e1;
                {borde}
            ">{valor_a_texto(M[i][j])}</td>
            """
        filas_html += f"<tr>{celdas}</tr>"

    titulo_html = f"<div style='font-weight:700; margin-bottom:8px;'>{titulo}</div>" if titulo else ""

    return f"""
    <div style="margin:10px 0 18px 0;">
        {titulo_html}
        <table style="border-collapse:collapse;">
            {filas_html}
        </table>
    </div>
    """



def html_gauss_jordan_inversa(nombre, A, R, pasos):
    bloques = []

    bloques.append(
        tarjeta_html(
            f"Inversa de {nombre} por Gauss-Jordan",
            "Se trabaja con la matriz aumentada <b>[A | I]</b>. "
            "Al transformar la parte izquierda en la identidad, la parte derecha se convierte en <b>A⁻¹</b>.",
            color="#d97706",
            fondo="#fff7ed"
        )
    )

    bloques.append(matriz_a_html(
        nombre=nombre,
        M=A,
        origen="Matriz original",
        dimensiones_txt=f"{len(A)}x{len(A[0])}"
    ))

    for k, paso in enumerate(pasos, start=1):
        bloques.append(f"""
        <div style="margin:14px 0; padding:12px; border:1px solid #fed7aa; border-radius:10px; background:#fffaf5;">
            <div style="font-weight:700; margin-bottom:8px;">Paso {k}: {paso['titulo']}</div>
            {html_matriz_aumentada(paso['matriz'], len(A))}
        </div>
        """)

    bloques.append(
        tarjeta_html(
            "Resultado final",
            f"<div>La matriz inversa obtenida es <b>{nombre}⁻¹</b>.</div>",
            color="#16a34a",
            fondo="#f0fdf4"
        )
    )

    bloques.append(matriz_a_html(
        nombre=nombre + "⁻¹",
        M=R,
        origen="Inversa calculada",
        dimensiones_txt=f"{len(R)}x{len(R[0])}"
    ))

    return "\n".join(bloques)


def html_cofactores_inversa(nombre, A, R, datos):
    detA = datos["det"]
    C = datos["cofactores"]
    Adj = datos["adjunta"]
    pasos = datos["pasos_cofactores"]

    bloques = []

    bloques.append(
        tarjeta_html(
            f"Inversa de {nombre} por cofactores",
            f"Se usa la fórmula <b>A⁻¹ = (1/det(A))·Adj(A)</b>, con <b>det(A) = {valor_a_texto(detA)}</b>.",
            color="#b45309",
            fondo="#fffbeb"
        )
    )

    bloques.append(matriz_a_html(
        nombre=nombre,
        M=A,
        origen="Matriz original",
        dimensiones_txt=f"{len(A)}x{len(A[0])}"
    ))

    bloques.append(tarjeta_html(
        "Paso 1: Determinante",
        f"<pre style='margin:0; white-space:pre-wrap;'>{datos['proc_det']}</pre>",
        color="#2563eb",
        fondo="#eff6ff"
    ))

    detalle = ""
    for paso in pasos:
        i = paso["i"] + 1
        j = paso["j"] + 1
        detalle += f"""
        <div style="margin:10px 0; padding:10px; border:1px solid #e5e7eb; border-radius:8px; background:#ffffff;">
            <div><b>c{i}{j}</b> = {valor_a_texto(paso['cofactor'])}</div>
            <div style="margin-top:6px;">Menor asociado:</div>
            <pre style="margin:6px 0 0 0;">{matriz_a_texto(paso['menor'])}</pre>
        </div>
        """

    bloques.append(f"""
    <details style="margin-top:12px;">
        <summary style="cursor:pointer; font-weight:700;">Ver cálculo de cofactores</summary>
        <div style="margin-top:10px;">{detalle}</div>
    </details>
    """)

    bloques.append(matriz_a_html(
        nombre="Cof(" + nombre + ")",
        M=C,
        origen="Matriz de cofactores",
        dimensiones_txt=f"{len(C)}x{len(C[0])}"
    ))

    bloques.append(matriz_a_html(
        nombre="Adj(" + nombre + ")",
        M=Adj,
        origen="Adjunta = (Cof(A))^T",
        dimensiones_txt=f"{len(Adj)}x{len(Adj[0])}"
    ))

    bloques.append(matriz_a_html(
        nombre=nombre + "⁻¹",
        M=R,
        origen=f"(1/{valor_a_texto(detA)})·Adj({nombre})",
        dimensiones_txt=f"{len(R)}x{len(R[0])}"
    ))

    return "\n".join(bloques)


def html_matriz_simple(M):
    filas_html = ""
    for fila in M:
        celdas = "".join(
            f"<td style='padding:6px 10px; border:1px solid #cbd5e1; text-align:center; font-family:monospace;'>{valor_a_texto(v)}</td>"
            for v in fila
        )
        filas_html += f"<tr>{celdas}</tr>"
    return f"<table style='border-collapse:collapse; margin:8px 0;'>{filas_html}</table>"




def formula_expansion_fila_html(A, fila, nombre='A'):
    if A is None or fila is None:
        return ""
    n = len(A)
    terminos_generales = [f"a<sub>{fila+1},{j+1}</sub>C<sub>{fila+1},{j+1}</sub>" for j in range(n)]
    terminos_caso = []
    for j in range(n):
        valor = valor_a_texto(A[fila][j])
        if sp.simplify(A[fila][j]) == 0:
            terminos_caso.append(f"{valor}·C<sub>{fila+1},{j+1}</sub>")
        else:
            terminos_caso.append(f"{valor}·C<sub>{fila+1},{j+1}</sub>")
    return (
        f"<div><b>Expansión por la fila {fila+1}:</b></div>"
        f"<div>det({nombre}) = " + " + ".join(terminos_generales) + "</div>"
        f"<div>det({nombre}) = " + " + ".join(terminos_caso) + "</div>"
    )

def detalle_termino_cofactor_html(fila, j, valor, signo, det_menor, cofactor, aporte):
    signo_txt = '+' if signo == 1 else '-'
    return f"""
    <div><b>Fórmula del cofactor:</b> C<sub>{fila+1},{j+1}</sub> = (-1)<sup>{fila+1}+{j+1}</sup> · det(M<sub>{fila+1},{j+1}</sub>)</div>
    <div>C<sub>{fila+1},{j+1}</sub> = {signo_txt} det(M<sub>{fila+1},{j+1}</sub>)</div>
    <div>C<sub>{fila+1},{j+1}</sub> = {signo_txt} ({valor_a_texto(det_menor)}) = {valor_a_texto(cofactor)}</div>
    <div style='margin-top:6px;'><b>Aporte al determinante:</b></div>
    <div>a<sub>{fila+1},{j+1}</sub> C<sub>{fila+1},{j+1}</sub> = {valor_a_texto(valor)} · {valor_a_texto(cofactor)} = {valor_a_texto(aporte)}</div>
    """


def html_det_cofactores_rec(datos, nombre='A', profundidad=0):
    # Compatibilidad con dos formatos de datos:
    # 1) formato antiguo: tipo, fila_expansion, matriz, resultado, terminos...
    # 2) formato nuevo: metodo, fila, det, procedimiento, terminos...
    tipo = datos.get('tipo') or datos.get('metodo')

    if tipo in ('1x1', 1):
        matriz = datos.get('matriz')
        resultado = datos.get('resultado', datos.get('det'))
        cuerpo = ""
        if matriz is not None:
            cuerpo += html_matriz_simple(matriz)
        cuerpo += f"<div><b>det({nombre}) = {valor_a_texto(resultado)}</b></div>"
        return tarjeta_html(
            f"Determinante {nombre}",
            cuerpo,
            color="#475569",
            fondo="#f8fafc"
        )

    if tipo in ('2x2', 2):
        # Soporta detalle antiguo o reconstrucción desde la matriz actual
        matriz = datos.get('matriz')
        detalle = datos.get('detalle')
        resultado = datos.get('resultado', datos.get('det'))

        if detalle is None and matriz is not None and len(matriz) == 2:
            a, b = matriz[0]
            c, d = matriz[1]
            detalle = {
                'a': a, 'b': b, 'c': c, 'd': d,
                'ad': sp.simplify(a*d),
                'bc': sp.simplify(b*c),
            }

        cuerpo = ""
        if matriz is not None:
            cuerpo += html_matriz_simple(matriz)
        if detalle is not None:
            cuerpo += f"""
            <div><b>Fórmula:</b> det({nombre}) = ad - bc</div>
            <div>det({nombre}) = ({valor_a_texto(detalle['a'])}·{valor_a_texto(detalle['d'])}) - ({valor_a_texto(detalle['b'])}·{valor_a_texto(detalle['c'])})</div>
            <div>det({nombre}) = {valor_a_texto(detalle['ad'])} - {valor_a_texto(detalle['bc'])}</div>
            """
        cuerpo += f"<div style='margin-top:8px; font-weight:700;'>det({nombre}) = {valor_a_texto(resultado)}</div>"
        return tarjeta_html(f"Menor {nombre} (2x2)", cuerpo, color="#475569", fondo="#f8fafc")


    # Formato nuevo simplificado
    if datos.get('metodo') == 'cofactores':
        fila = datos.get('fila')
        terminos = datos.get('terminos', [])
        resultado = datos.get('det')
        procedimiento = datos.get('procedimiento', '')
        matriz = datos.get('matriz')

        intro_cuerpo = ""
        if matriz is not None:
            intro_cuerpo += "<div style='margin-bottom:8px;'><b>Matriz original:</b></div>"
            intro_cuerpo += html_matriz_simple(matriz)

        intro_cuerpo += (
            f"<div><b>Método usado:</b> expansión por cofactores.</div>"
            + (f"<div><b>Fila elegida:</b> {fila + 1}</div>" if fila is not None else "")
            + "<div><b>Fórmula del cofactor:</b> C<sub>i,j</sub> = (-1)<sup>i+j</sup> · det(M<sub>i,j</sub>)</div>"
            + "<div><b>Fórmula del determinante por cofactores:</b> det(A) = Σ a<sub>i,j</sub> C<sub>i,j</sub></div>"
        )
        if matriz is not None and fila is not None:
            intro_cuerpo += "<div style='margin-top:8px;'>" + formula_expansion_fila_html(matriz, fila, nombre) + "</div>"
        if procedimiento:
            intro_cuerpo += f"<div style='margin-top:8px;'>{bloque_texto_pre(procedimiento)}</div>"

        intro = tarjeta_html(
            f"Determinante de {nombre}",
            intro_cuerpo,
            color="#2563eb",
            fondo="#eff6ff"
        )

        bloques = [intro]
        for idx, termino in enumerate(terminos, start=1):
            j = termino.get('columna')
            valor = termino.get('valor')
            if termino.get('anulado'):
                cuerpo = f"""
                <div><b>Columna:</b> {j + 1}</div>
                <div><b>Elemento:</b> a<sub>{fila+1},{j+1}</sub> = {valor_a_texto(valor)}</div>
                <div><b>Cofactor asociado:</b> C<sub>{fila+1},{j+1}</sub></div>
                <div><b>Aporte:</b> a<sub>{fila+1},{j+1}</sub> C<sub>{fila+1},{j+1}</sub> = 0 · C<sub>{fila+1},{j+1}</sub> = 0</div>
                <div style='margin-top:6px;'>Este término no aporta porque el elemento de la fila elegida es 0.</div>
                """
                bloques.append(tarjeta_html(
                    f"Término {idx}",
                    cuerpo,
                    color="#94a3b8",
                    fondo="#f8fafc"
                ))
                continue

            signo = termino.get('signo')
            menor = termino.get('menor')
            det_menor = termino.get('det_menor')
            cofactor = termino.get('cofactor')
            aporte = termino.get('aporte')

            cuerpo = f"""
            <div><b>Columna:</b> {j + 1}</div>
            <div><b>Elemento de la fila elegida:</b> a<sub>{fila+1},{j+1}</sub> = {valor_a_texto(valor)}</div>
            <div><b>Menor asociado:</b> M<sub>{fila+1},{j+1}</sub></div>
            {html_matriz_simple(menor) if menor is not None else ''}
            <div><b>Determinante del menor:</b> det(M<sub>{fila+1},{j+1}</sub>) = {valor_a_texto(det_menor)}</div>
            {detalle_termino_cofactor_html(fila, j, valor, signo, det_menor, cofactor, aporte)}
            """
            bloques.append(tarjeta_html(
                f"Término {idx}",
                cuerpo,
                color="#d97706",
                fondo="#fff7ed"
            ))

        aportes = []
        for t in terminos:
            if t.get('anulado'):
                aportes.append("0")
            else:
                aportes.append(valor_a_texto(t['aporte']))
        expresion = ' + '.join(aportes) if aportes else '0'
        bloques.append(tarjeta_html(
            "Resultado final",
            f"<div><b>Suma de aportes:</b> {expresion}</div><div style='margin-top:6px;'>Por lo tanto,</div><div style='margin-top:6px; font-size:18px;'><b>det({nombre}) = {valor_a_texto(resultado)}</b></div>",
            color="#16a34a",
            fondo="#f0fdf4"
        ))
        return ''.join(bloques)

    # Formato antiguo recursivo
    fila = datos['fila_expansion']
    n = len(datos['matriz'])
    intro = tarjeta_html(
        f"Determinante de {nombre} ({n}x{n})",
        f"<div><b>Se expande por la fila {fila}</b> porque reduce el cálculo.</div>{html_matriz_simple(datos['matriz'])}",
        color="#2563eb",
        fondo="#eff6ff"
    )

    bloques = [intro]
    for termino in datos['terminos']:
        i, j = termino['fila'], termino['col']
        if termino['cero']:
            bloques.append(tarjeta_html(
                f"Término a({i},{j})",
                f"<div>a<sub>{i}{j}</sub> = 0, por lo tanto este término <b>no aporta</b>.</div>",
                color="#94a3b8",
                fondo="#f8fafc"
            ))
            continue
        signo_txt = '+' if termino['signo'] == 1 else '-'
        cuerpo = f"""
        <div><b>Elemento:</b> a<sub>{i}{j}</sub> = {valor_a_texto(termino['valor'])}</div>
        <div><b>Signo del cofactor:</b> {signo_txt}</div>
        <div><b>Menor</b> M<sub>{i}{j}</sub>:</div>
        {html_matriz_simple(termino['menor'])}
        <div><b>det(M<sub>{i}{j}</sub>) = {valor_a_texto(termino['det_menor'])}</b></div>
        <div style='margin-top:6px;'><b>Aporte:</b> ({valor_a_texto(termino['signo'])})·({valor_a_texto(termino['valor'])})·({valor_a_texto(termino['det_menor'])}) = <b>{valor_a_texto(termino['aporte'])}</b></div>
        """
        detalle = html_det_cofactores_rec(termino['datos_menor'], nombre=f"M{i}{j}", profundidad=profundidad+1)
        bloques.append(f"<div style='margin:14px 0 14px {20 if profundidad else 0}px;'>{tarjeta_html(f'Término ({i},{j})', cuerpo, color='#d97706', fondo='#fff7ed')}{detalle}</div>")

    expresion = ' + '.join(valor_a_texto(t['aporte']) for t in datos['terminos'] if not t['cero']) or '0'
    bloques.append(tarjeta_html(
        "Resultado final",
        f"<div><b>Suma de aportes:</b> {expresion}</div><div style='margin-top:6px; font-size:18px;'><b>det({nombre}) = {valor_a_texto(datos['resultado'])}</b></div>",
        color="#16a34a",
        fondo="#f0fdf4"
    ))
    return ''.join(bloques)


def html_determinante_4x4(A, nombre='A'):
    datos = datos_determinante_cofactores(A)
    return html_det_cofactores_rec(datos, nombre=nombre)


def titulo_seccion_html(titulo, subtitulo=""):
    sub = f"<div style='color:#475569; margin-top:4px;'>{subtitulo}</div>" if subtitulo else ""
    return f"""
    <div style="
        background: linear-gradient(135deg, #eff6ff 0%, #f8fafc 100%);
        border:1px solid #dbeafe;
        border-left:6px solid #2563eb;
        border-radius:12px;
        padding:14px 16px;
        margin:4px 0 12px 0;
    ">
        <div style="font-size:20px; font-weight:800; color:#0f172a;">{titulo}</div>
        {sub}
    </div>
    """

def tip_uso_html(ejercicios, para_que_sirve, notas=""):
    notas_html = f"<div style='margin-top:8px; color:#334155;'><b>Pista:</b> {notas}</div>" if notas else ""
    return f"""
    <div style="
        background:#fffbeb;
        border:1px solid #fde68a;
        border-left:6px solid #f59e0b;
        border-radius:10px;
        padding:12px 14px;
        margin:8px 0 14px 0;
    ">
        <div style="font-weight:800; color:#92400e; margin-bottom:6px;">Tips de uso</div>
        <div><b>Ejercicios de la guía:</b> {ejercicios}</div>
        <div style='margin-top:6px;'><b>Úsalo para:</b> {para_que_sirve}</div>
        {notas_html}
    </div>
    """

def caja_salida(titulo="Mensajes"):
    return widgets.HTML(f"<div style='font-weight:700; margin:10px 0 6px 0; color:#334155;'>{titulo}</div>")


## Interfaz separada por bloques

Se mantiene la lógica original y los **nombres automáticos**. Solo se separó la interfaz en celdas para ejecutar y probar cada parte con más facilidad.

### Salidas

Bloque interno de la interfaz.

In [ ]:
# ---------- SALIDAS ----------
salida_crear_manual = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='90px'
))

salida_crear_formula = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='90px'
))

salida_transformar = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='90px'
))

salida_operar = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='90px'
))

salida_det = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='110px'
))

salida_guardadas = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='250px'
))

salida_proc = widgets.Output(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', min_height='220px'
))

### Estado Ui

Bloque interno de la interfaz.

In [ ]:
# ---------- ESTADO UI ----------
grid_inputs_manual = []

### Helpers

Bloque interno de la interfaz.

In [ ]:
# ---------- HELPERS ----------
def lista_nombres():
    return nombres_matrices()

def refrescar_panel_guardadas():
    with salida_guardadas:
        clear_output()
        display(renderizar_panel_matrices())


def refrescar_selectores():
    opciones = lista_nombres()

    dropdowns = []
    nombres_dropdown = [
        'selector_transformar', 'selector_op1', 'selector_op2', 'selector_det',
        'selector_proc', 'selector_eliminar', 'selector_ax_A', 'selector_ax_B',
        'selector_xa_A', 'selector_xa_B', 'selector_prob_P', 'selector_prob_Q',
        'selector_auto_inv', 'selector_poly_A'
    ]

    for nombre_dd in nombres_dropdown:
        if nombre_dd in globals():
            dropdowns.append(globals()[nombre_dd])

    for dd in dropdowns:
        valor_actual = dd.value if getattr(dd, "options", None) else None
        dd.options = opciones
        if opciones:
            dd.value = valor_actual if valor_actual in opciones else opciones[0]
        else:
            dd.options = []

def mostrar_info_matriz(nombre, salida_widget):
    with salida_widget:
        clear_output()
        if not nombre:
            display(HTML(html_info("No hay matriz seleccionada.")))
            return
        info = obtener_info(nombre)
        display(HTML(html_procedimiento_matriz(nombre, info)))

### Crear Manual

Usa este bloque para cargar matrices base de cualquier ejercicio. Mantiene los nombres automáticos A, B, C, ...

In [ ]:
# ---------- CREAR MANUAL ----------

filas_manual = widgets.BoundedIntText(
    value=2, min=1, max=8, description='Filas:',
    layout=widgets.Layout(width='150px')
)

cols_manual = widgets.BoundedIntText(
    value=2, min=1, max=8, description='Cols:',
    layout=widgets.Layout(width='150px')
)

btn_generar_grid_manual = widgets.Button(
    description='Generar grilla',
    button_style='info',
    layout=widgets.Layout(width='140px')
)

btn_crear_manual = widgets.Button(
    description='Crear matriz',
    button_style='success',
    layout=widgets.Layout(width='130px')
)

contenedor_grid_manual = widgets.VBox(layout=widgets.Layout(
    border='1px solid #d0d7de', padding='10px', margin='10px 0'
))

selector_eliminar = widgets.Dropdown(
    options=[],
    description='Eliminar:',
    layout=widgets.Layout(width='240px')
)

btn_eliminar = widgets.Button(
    description='Eliminar matriz',
    button_style='danger',
    layout=widgets.Layout(width='140px')
)

def generar_grid_manual(filas, columnas):
    global grid_inputs_manual
    grid_inputs_manual = []

    filas_widgets = []
    for i in range(filas):
        fila_datos = []
        fila_widgets = []
        for j in range(columnas):
            txt = widgets.Text(value="0", layout=widgets.Layout(width="72px"))
            fila_datos.append(txt)
            fila_widgets.append(txt)
        grid_inputs_manual.append(fila_datos)
        filas_widgets.append(widgets.HBox(fila_widgets, layout=widgets.Layout(margin='2px 0')))
    contenedor_grid_manual.children = tuple(filas_widgets)

def leer_grid_manual():
    M = []
    for fila in grid_inputs_manual:
        nueva_fila = []
        for celda in fila:
            nueva_fila.append(celda.value.strip())
        M.append(nueva_fila)
    return M

def accion_generar_grid_manual(_):
    with salida_crear_manual:
        clear_output()
        generar_grid_manual(filas_manual.value, cols_manual.value)
        display(HTML(html_info(f"Se generó una grilla de <b>{filas_manual.value}x{cols_manual.value}</b>. Ahora completa los valores y pulsa <b>Crear matriz</b>.")))

def accion_crear_manual(_):
    with salida_crear_manual:
        clear_output()
        try:
            nombre = crear_matriz_manual(leer_grid_manual())
            info = obtener_info(nombre)
            display(HTML(
                tarjeta_html(
                    "Matriz creada",
                    f"Se creó la matriz <b>{nombre}</b> con dimensiones <b>{info['dimensiones'][0]}x{info['dimensiones'][1]}</b>.",
                    color="#16a34a", fondo="#f0fdf4"
                ) + matriz_a_html(nombre, info["matriz"], info["origen"], f"{info['dimensiones'][0]}x{info['dimensiones'][1]}")
            ))
            refrescar_panel_guardadas()
            refrescar_selectores()
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_eliminar(_):
    with salida_crear_manual:
        clear_output()
        if not selector_eliminar.options:
            display(HTML(html_info("No hay matrices para eliminar.")))
            return
        nombre = selector_eliminar.value
        eliminar_matriz(nombre)
        display(HTML(tarjeta_html("Matriz eliminada", f"Se eliminó la matriz <b>{nombre}</b>.", color="#dc2626", fondo="#fef2f2")))
        refrescar_panel_guardadas()
        refrescar_selectores()

btn_generar_grid_manual.on_click(accion_generar_grid_manual)
btn_crear_manual.on_click(accion_crear_manual)
btn_eliminar.on_click(accion_eliminar)

tab_crear_manual = widgets.VBox([
    widgets.HTML(titulo_seccion_html("Crear matriz manualmente", "Entrada rápida para matrices de cualquier ejercicio")),
    widgets.HTML(tip_uso_html(
        "P1.1, P1.2, P1.4, P1.5, P1.7, P1.8, P1.9, P1.10 · P2.1, P2.6, P2.7, P2.8, P2.9, P2.10",
        "cargar matrices tal como aparecen en la guía y que el notebook les asigne nombres automáticos A, B, C...",
        "Primero genera la grilla. Luego rellena valores y crea la matriz."
    )),
    widgets.HBox([filas_manual, cols_manual, btn_generar_grid_manual], layout=widgets.Layout(gap='10px')),
    contenedor_grid_manual,
    btn_crear_manual,
    widgets.HTML("<hr>"),
    widgets.HTML("<div style='font-weight:700; color:#334155;'>Eliminar matriz guardada</div>"),
    widgets.HBox([selector_eliminar, btn_eliminar], layout=widgets.Layout(gap='10px')),
    caja_salida(),
    salida_crear_manual
])


### Crear Por Fórmula

Úsalo para ejercicios con definición por fórmula, como P1.3(a)-(b).

In [ ]:
# ---------- CREAR POR FÓRMULA ----------
filas_formula = widgets.BoundedIntText(
    value=3, min=1, max=10, description='Filas:',
    layout=widgets.Layout(width='150px')
)

cols_formula = widgets.BoundedIntText(
    value=3, min=1, max=10, description='Cols:',
    layout=widgets.Layout(width='150px')
)

txt_formula = widgets.Text(
    value='i + j',
    description='Fórmula:',
    layout=widgets.Layout(width='360px')
)

btn_crear_formula = widgets.Button(
    description='Crear por fórmula',
    button_style='success',
    layout=widgets.Layout(width='150px')
)

def accion_crear_formula(_):
    with salida_crear_formula:
        clear_output()
        try:
            nombre = crear_matriz_por_formula(
                filas=filas_formula.value,
                columnas=cols_formula.value,
                formula=txt_formula.value.strip()
            )
            info = obtener_info(nombre)
            display(HTML(
                tarjeta_html(
                    "Matriz creada por fórmula",
                    f"Se creó <b>{nombre}</b> usando la fórmula <code>{txt_formula.value.strip()}</code>.",
                    color="#16a34a", fondo="#f0fdf4"
                ) + matriz_a_html(nombre, info["matriz"], info["origen"], f"{info['dimensiones'][0]}x{info['dimensiones'][1]}")
                + bloque_texto_pre(info["procedimiento"])
            ))
            refrescar_panel_guardadas()
            refrescar_selectores()
        except Exception as e:
            display(HTML(html_error(str(e))))

btn_crear_formula.on_click(accion_crear_formula)

tab_crear_formula = widgets.VBox([
    widgets.HTML(titulo_seccion_html("Crear matriz por fórmula", "Generación automática desde expresiones en i y j")),
    widgets.HTML(tip_uso_html(
        "P1.3(a), P1.3(b) y cualquier ejercicio donde la matriz venga dada por aᵢⱼ o bᵢⱼ",
        "construir matrices usando variables <b>i</b> y <b>j</b> sin escribir las entradas una por una",
        "Ejemplos: <code>i**2 - i*j</code>, <code>i + 2*j</code>, <code>2*i - j</code>."
    )),
    widgets.HBox([filas_formula, cols_formula], layout=widgets.Layout(gap='10px')),
    txt_formula,
    btn_crear_formula,
    caja_salida(),
    salida_crear_formula
])


### Transformar

Úsalo para transpuesta, escalar, potencia e inversa. Sirve para P1.4, P2.6, P2.7(a), P2.8.

In [ ]:
# ---------- TRANSFORMAR ----------
selector_transformar = widgets.Dropdown(
    options=[],
    description='Base:',
    layout=widgets.Layout(width='240px')
)

chk_T = widgets.Checkbox(value=False, description='Transpuesta')
chk_kA = widgets.Checkbox(value=False, description='Escalar')
txt_escalar = widgets.Text(value='2', description='k:', layout=widgets.Layout(width='170px'))
chk_pot = widgets.Checkbox(value=False, description='Potencia')
int_pot = widgets.BoundedIntText(value=2, min=1, max=10, description='n:', layout=widgets.Layout(width='170px'))
chk_inv = widgets.Checkbox(value=False, description='Inversa')

btn_transformar = widgets.Button(
    description='Generar transformaciones',
    button_style='warning',
    layout=widgets.Layout(width='190px')
)

def accion_transformar(_):
    with salida_transformar:
        clear_output()
        try:
            if not selector_transformar.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return

            nombre = selector_transformar.value
            creadas = transformar_y_guardar(
                nombre_base=nombre,
                hacer_transpuesta=chk_T.value,
                hacer_escalar=chk_kA.value,
                escalar=txt_escalar.value.strip(),
                hacer_potencia=chk_pot.value,
                exponente=int_pot.value,
                hacer_inversa=chk_inv.value
            )

            if creadas:
                display(HTML(html_resultado_transformacion(creadas)))
                for x in creadas:
                    info = obtener_info(x)
                    display(HTML(matriz_a_html(
                        nombre=x,
                        M=info["matriz"],
                        origen=info["origen"],
                        dimensiones_txt=f"{info['dimensiones'][0]}x{info['dimensiones'][1]}"
                    )))
            else:
                display(HTML(html_info("No seleccionaste ninguna transformación.")))

            refrescar_panel_guardadas()
            refrescar_selectores()

        except Exception as e:
            display(HTML(html_error(str(e))))

btn_transformar.on_click(accion_transformar)

tab_transformar = widgets.VBox([
    widgets.HTML(titulo_seccion_html("Transformar matrices", "Aplica transpuesta, escalar, potencia o inversa y guarda el resultado")),
    widgets.HTML(tip_uso_html(
        "P1.2, P1.4, P1.5, P1.7, P1.8 · P2.6, P2.7, P2.8",
        "obtener Aᵀ, kA, Aⁿ o A⁻¹ sin perder el nombre original",
        "Marca una o varias casillas y el notebook guardará las nuevas matrices con nombre automático."
    )),
    selector_transformar,
    chk_T,
    widgets.HBox([chk_kA, txt_escalar], layout=widgets.Layout(gap='10px')),
    widgets.HBox([chk_pot, int_pot], layout=widgets.Layout(gap='10px')),
    chk_inv,
    btn_transformar,
    caja_salida(),
    salida_transformar
])


### Operar

Úsalo para suma, resta y producto. Sirve para P1.1, P1.2 y pasos intermedios de muchos otros.

In [ ]:
# ---------- OPERAR ----------
selector_op1 = widgets.Dropdown(
    options=[],
    description='Matriz 1:',
    layout=widgets.Layout(width='240px')
)

selector_operacion = widgets.Dropdown(
    options=['Suma', 'Resta', 'Producto', 'Escalar'],
    description='Op:',
    layout=widgets.Layout(width='240px')
)

selector_op2 = widgets.Dropdown(
    options=[],
    description='Matriz 2:',
    layout=widgets.Layout(width='240px')
)

txt_escalar_op = widgets.Text(
    value='2',
    description='k:',
    layout=widgets.Layout(width='170px')
)

btn_operar = widgets.Button(
    description='Ejecutar operación',
    button_style='success',
    layout=widgets.Layout(width='160px')
)

def actualizar_visibilidad_operacion(*args):
    op = selector_operacion.value
    selector_op2.layout.display = None if op in ['Suma', 'Resta', 'Producto'] else 'none'
    txt_escalar_op.layout.display = None if op == 'Escalar' else 'none'

selector_operacion.observe(actualizar_visibilidad_operacion, names='value')

def accion_operar(_):
    with salida_operar:
        clear_output()
        try:
            if not selector_op1.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return

            op = selector_operacion.value
            nombre1 = selector_op1.value

            if op == 'Escalar':
                nombre_res = operar_y_guardar(nombre1, op, escalar=txt_escalar_op.value.strip())
            else:
                nombre2 = selector_op2.value
                nombre_res = operar_y_guardar(nombre1, op, nombre2=nombre2)

            info = obtener_info(nombre_res)
            display(HTML(html_resultado_operacion(nombre_res, info)))

            refrescar_panel_guardadas()
            refrescar_selectores()

        except Exception as e:
            display(HTML(html_error(str(e))))

btn_operar.on_click(accion_operar)

tab_operar = widgets.VBox([
    widgets.HTML(titulo_seccion_html("Operaciones entre matrices", "Suma, resta, producto y producto por escalar")),
    widgets.HTML(tip_uso_html(
        "P1.1, P1.2, P1.3(b), P1.4, P1.5, P1.6, P1.7, P1.10 · P2.7(b), P2.9",
        "combinar matrices ya creadas y guardar automáticamente el resultado con un nombre nuevo",
        "En producto, revisa que las dimensiones sean compatibles. En escalar, solo usa la primera matriz."
    )),
    selector_op1,
    selector_operacion,
    selector_op2,
    txt_escalar_op,
    btn_operar,
    caja_salida(),
    salida_operar
])

actualizar_visibilidad_operacion()


### Det / Inversa

Úsalo para determinantes e inversas con procedimiento. Sirve para P2.1, P2.2, P2.3, P2.4, P2.5, P2.7, P2.8.

In [ ]:
# ---------- DET / INVERSA ----------
selector_det = widgets.Dropdown(
    options=[],
    description='Matriz:',
    layout=widgets.Layout(width='240px')
)

selector_metodo_inversa = widgets.Dropdown(
    options=[
        ("Operaciones elementales (Gauss-Jordan)", "gauss"),
        ("Matriz de cofactores", "cofactores")
    ],
    value="gauss",
    description="Método:",
    layout=widgets.Layout(width='360px')
)

btn_det = widgets.Button(
    description='Calcular determinante',
    button_style='info',
    layout=widgets.Layout(width='170px')
)

btn_inv = widgets.Button(
    description='Calcular inversa',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

def accion_det(_):
    with salida_det:
        clear_output()
        try:
            if not selector_det.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return

            nombre = selector_det.value
            A = obtener_matriz(nombre)
            det, proc = determinante_matriz(A)

            display(HTML(tarjeta_html(
                "Determinante calculado",
                f"<b>det({nombre}) = {valor_a_texto(det)}</b>",
                color="#2563eb",
                fondo="#eff6ff"
            )))

            if es_cuadrada(A) and len(A) == 3:
                display(HTML(html_sarrus_3x3(A, nombre)))
            elif es_cuadrada(A) and len(A) == 4:
                display(HTML(html_determinante_4x4(A, nombre)))
            else:
                display(HTML(bloque_texto_pre(proc)))

        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_inv(_):
    with salida_det:
        clear_output()
        try:
            if not selector_det.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return

            nombre = selector_det.value
            metodo = selector_metodo_inversa.value
            A = obtener_matriz(nombre)

            R, proc_texto, datos = inversa_matriz(A, metodo=metodo)

            if metodo == "gauss":
                html_proc = html_gauss_jordan_inversa(nombre, A, R, datos["pasos"])
                origen = f"Inversa de {nombre} por operaciones elementales"
                resumen = f"Se calculó y guardó la inversa <b>{nombre_inversa(nombre)}</b> usando Gauss-Jordan."
                tipo = "inversa-gauss"
            else:
                html_proc = html_cofactores_inversa(nombre, A, R, datos)
                origen = f"Inversa de {nombre} por matriz de cofactores"
                resumen = f"Se calculó y guardó la inversa <b>{nombre_inversa(nombre)}</b> usando cofactores."
                tipo = "inversa-cofactores"

            nombre_inv = nombre_inversa(nombre)

            registrar_resultado(
                nombre_inv,
                R,
                origen,
                proc_texto,
                procedimiento_html=html_proc,
                resumen=resumen,
                tipo=tipo
            )

            display(HTML(html_proc))
            display(HTML(tarjeta_html(
                "Matriz guardada",
                f"La inversa se guardó como <b>{nombre_inv}</b>.",
                color="#16a34a",
                fondo="#f0fdf4"
            )))

            refrescar_panel_guardadas()
            refrescar_selectores()

        except Exception as e:
            display(HTML(html_error(str(e))))

btn_det.on_click(accion_det)
btn_inv.on_click(accion_inv)

tab_det = widgets.VBox([
    widgets.HTML("<h3>Determinante e inversa</h3>"),
    selector_det,
    selector_metodo_inversa,
    widgets.HBox([btn_det, btn_inv], layout=widgets.Layout(gap='10px')),
    widgets.HTML("<b>Mensajes</b>"),
    salida_det
])

### Guardadas

Panel para revisar matrices registradas y sus nombres automáticos.

In [ ]:
# ---------- GUARDADAS ----------
tab_guardadas = widgets.VBox([
    widgets.HTML("<h3>Matrices guardadas</h3>"),
    salida_guardadas
])

### Procedimientos

Panel para ver el procedimiento guardado de cualquier matriz resultado.

In [ ]:
# ---------- PROCEDIMIENTOS ----------
selector_proc = widgets.Dropdown(
    options=[],
    description='Matriz:',
    layout=widgets.Layout(width='240px')
)

btn_ver_proc = widgets.Button(
    description='Ver procedimiento',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

btn_ver_hist = widgets.Button(
    description='Ver historial',
    layout=widgets.Layout(width='120px')
)


def accion_ver_proc(_):
    with salida_proc:
        clear_output()
        try:
            if not selector_proc.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return

            nombre = selector_proc.value
            info = obtener_info(nombre)
            display(HTML(html_procedimiento_matriz(nombre, info)))

        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_ver_hist(_):
    with salida_proc:
        clear_output()
        display(HTML(html_historial()))

btn_ver_proc.on_click(accion_ver_proc)

btn_ver_hist.on_click(accion_ver_hist)

tab_proc = widgets.VBox([
    widgets.HTML("<h3>Procedimientos</h3>"),
    selector_proc,
    widgets.HBox([btn_ver_proc, btn_ver_hist], layout=widgets.Layout(gap='10px')),
    salida_proc
])

### Resolvedores / Tests

Bloque para probar AX=B, XA=B, identidades, parámetros, probabilidad y casos especiales.

In [ ]:
# ---------- RESOLVEDORES / TESTS ----------
salida_resolver_ax = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='110px'))
salida_resolver_xa = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='110px'))
salida_ec_lineal = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='110px'))
salida_identidad = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))
salida_param_det = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))
salida_prob = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))
salida_casos = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))
salida_auto_inv = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))
salida_poly = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))
salida_vars = widgets.Output(layout=widgets.Layout(border='1px solid #d0d7de', padding='10px', min_height='100px'))

selector_ax_A = widgets.Dropdown(options=[], description='A:', layout=widgets.Layout(width='240px'))
selector_ax_B = widgets.Dropdown(options=[], description='B:', layout=widgets.Layout(width='240px'))
selector_metodo_ax = widgets.Dropdown(
    options=[('Gauss-Jordan', 'gauss'), ('Cofactores', 'cofactores')],
    value='gauss', description='Método:', layout=widgets.Layout(width='260px')
)
txt_nombre_ax = widgets.Text(value='X_AX', description='Guardar:', layout=widgets.Layout(width='240px'))
btn_resolver_ax = widgets.Button(description='Resolver AX = B', button_style='success', layout=widgets.Layout(width='170px'))

selector_xa_A = widgets.Dropdown(options=[], description='A:', layout=widgets.Layout(width='240px'))
selector_xa_B = widgets.Dropdown(options=[], description='B:', layout=widgets.Layout(width='240px'))
selector_metodo_xa = widgets.Dropdown(
    options=[('Gauss-Jordan', 'gauss'), ('Cofactores', 'cofactores')],
    value='gauss', description='Método:', layout=widgets.Layout(width='260px')
)
txt_nombre_xa = widgets.Text(value='X_XA', description='Guardar:', layout=widgets.Layout(width='240px'))
btn_resolver_xa = widgets.Button(description='Resolver XA = B', button_style='success', layout=widgets.Layout(width='170px'))

txt_eq_izq = widgets.Text(value='3X - 2A', description='Izq:', layout=widgets.Layout(width='320px'))
txt_eq_der = widgets.Text(value='B', description='Der:', layout=widgets.Layout(width='320px'))
txt_nombre_eq = widgets.Text(value='X_EQ', description='Guardar:', layout=widgets.Layout(width='240px'))
btn_resolver_eq = widgets.Button(description='Resolver ecuación', button_style='success', layout=widgets.Layout(width='170px'))

txt_id_izq = widgets.Text(value='(A+B)**2', description='Izq:', layout=widgets.Layout(width='320px'))
txt_id_der = widgets.Text(value='A**2 + B**2', description='Der:', layout=widgets.Layout(width='320px'))
btn_verificar_id = widgets.Button(description='Verificar identidad', button_style='info', layout=widgets.Layout(width='170px'))

txt_expr_param = widgets.Text(value='A - t*I', description='Matriz:', layout=widgets.Layout(width='320px'))
txt_var_param = widgets.Text(value='t', description='Variable:', layout=widgets.Layout(width='180px'))
txt_valor_param = widgets.Text(value='0', description='Objetivo:', layout=widgets.Layout(width='180px'))
btn_resolver_param = widgets.Button(description='Resolver parámetro', button_style='warning', layout=widgets.Layout(width='170px'))

selector_prob_P = widgets.Dropdown(options=[], description='P:', layout=widgets.Layout(width='240px'))
selector_prob_Q = widgets.Dropdown(options=[], description='Q:', layout=widgets.Layout(width='240px'))
btn_validar_P = widgets.Button(description='¿P es probabilidad?', layout=widgets.Layout(width='170px'))
btn_validar_PQ = widgets.Button(description='¿PQ es probabilidad?', layout=widgets.Layout(width='170px'))

txt_nombre_casos = widgets.Text(value='C', description='Nombre:', layout=widgets.Layout(width='220px'))
filas_casos = widgets.BoundedIntText(value=3, min=1, max=8, description='Filas:', layout=widgets.Layout(width='160px'))
cols_casos = widgets.BoundedIntText(value=3, min=1, max=8, description='Cols:', layout=widgets.Layout(width='160px'))
txt_cond_casos = widgets.Text(value='i <= j', description='Condición:', layout=widgets.Layout(width='260px'))
txt_si_casos = widgets.Text(value='i + j', description='Si:', layout=widgets.Layout(width='220px'))
txt_no_casos = widgets.Text(value='0', description='No:', layout=widgets.Layout(width='220px'))
btn_crear_casos = widgets.Button(description='Crear por casos', button_style='success', layout=widgets.Layout(width='150px'))


selector_auto_inv = widgets.Dropdown(options=[], description='A:', layout=widgets.Layout(width='240px'))
btn_auto_inv = widgets.Button(description='Verificar A⁻¹ = A', button_style='info', layout=widgets.Layout(width='170px'))

selector_poly_A = widgets.Dropdown(options=[], description='A:', layout=widgets.Layout(width='240px'))
txt_var_b = widgets.Text(value='b', description='Var 1:', layout=widgets.Layout(width='160px'))
txt_var_c = widgets.Text(value='c', description='Var 2:', layout=widgets.Layout(width='160px'))
btn_poly = widgets.Button(description='Resolver b,c', button_style='warning', layout=widgets.Layout(width='150px'))

txt_vars_lista = widgets.Text(value='x,y', description='Variables:', layout=widgets.Layout(width='220px'))
txt_vars_izq = widgets.Text(value='Matrix([[1,-1],[3,2]])*col(x,y)', description='Izq:', layout=widgets.Layout(width='420px'))
txt_vars_der = widgets.Text(value='Matrix([[1,x],[y,-1]])*col(3,2)', description='Der:', layout=widgets.Layout(width='420px'))
btn_resolver_vars = widgets.Button(description='Resolver variables', button_style='warning', layout=widgets.Layout(width='170px'))


def _datos_registrados():
    return {nombre: obtener_matriz(nombre) for nombre in nombres_matrices()}

def accion_resolver_ax(_):
    with salida_resolver_ax:
        clear_output()
        try:
            if not selector_ax_A.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return
            nombreA = selector_ax_A.value
            nombreB = selector_ax_B.value
            A = obtener_matriz(nombreA)
            B = obtener_matriz(nombreB)
            metodo = selector_metodo_ax.value
            X, proc, datos = resolver_AX_B(A, B, metodo=metodo)
            nombre_res = txt_nombre_ax.value.strip() or 'X_AX'
            resumen = f"Se resolvió <b>{nombreA}X = {nombreB}</b> y se guardó la solución como <b>{nombre_res}</b>."
            html_extra = html_gauss_jordan_inversa(nombreA, A, datos['inversa'], datos['datos_inversa']['pasos']) if metodo == 'gauss' else html_cofactores_inversa(nombreA, A, datos['inversa'], datos['datos_inversa'])
            proc_html = (
                tarjeta_html("Resolución de AX = B",
                    f"Se usa la fórmula <b>X = {nombreA}⁻¹{nombreB}</b>.",
                    color="#0f766e", fondo="#f0fdfa")
                + html_extra
                + matriz_a_html(nombre_res, X, f"Solución de {nombreA}X={nombreB}", f"{len(X)}x{len(X[0])}")
            )
            registrar_resultado(
                nombre_res, X, f"Solución de {nombreA}X = {nombreB}", proc,
                procedimiento_html=proc_html, resumen=resumen, tipo='resolver-AX=B'
            )
            info = obtener_info(nombre_res)
            display(HTML(html_resultado_operacion(nombre_res, info)))
            refrescar_panel_guardadas()
            refrescar_selectores()
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_resolver_xa(_):
    with salida_resolver_xa:
        clear_output()
        try:
            if not selector_xa_A.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return
            nombreA = selector_xa_A.value
            nombreB = selector_xa_B.value
            A = obtener_matriz(nombreA)
            B = obtener_matriz(nombreB)
            metodo = selector_metodo_xa.value
            X, proc, datos = resolver_XA_B(A, B, metodo=metodo)
            nombre_res = txt_nombre_xa.value.strip() or 'X_XA'
            resumen = f"Se resolvió <b>X{nombreA} = {nombreB}</b> y se guardó la solución como <b>{nombre_res}</b>."
            html_extra = html_gauss_jordan_inversa(nombreA, A, datos['inversa'], datos['datos_inversa']['pasos']) if metodo == 'gauss' else html_cofactores_inversa(nombreA, A, datos['inversa'], datos['datos_inversa'])
            proc_html = (
                tarjeta_html("Resolución de XA = B",
                    f"Se usa la fórmula <b>X = {nombreB}{nombreA}⁻¹</b>.",
                    color="#0f766e", fondo="#f0fdfa")
                + html_extra
                + matriz_a_html(nombre_res, X, f"Solución de X{nombreA}={nombreB}", f"{len(X)}x{len(X[0])}")
            )
            registrar_resultado(
                nombre_res, X, f"Solución de X{nombreA} = {nombreB}", proc,
                procedimiento_html=proc_html, resumen=resumen, tipo='resolver-XA=B'
            )
            info = obtener_info(nombre_res)
            display(HTML(html_resultado_operacion(nombre_res, info)))
            refrescar_panel_guardadas()
            refrescar_selectores()
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_resolver_ec_lineal(_):
    with salida_ec_lineal:
        clear_output()
        try:
            datos = _datos_registrados()
            X, proc, extra = resolver_ecuacion_matricial_lineal(txt_eq_izq.value.strip(), txt_eq_der.value.strip(), datos)
            nombre_res = txt_nombre_eq.value.strip() or 'X_EQ'
            resumen = f"Se resolvió la ecuación <b>{txt_eq_izq.value.strip()} = {txt_eq_der.value.strip()}</b>."
            proc_html = (
                tarjeta_html("Ecuación matricial lineal", resumen, color="#7c3aed", fondo="#faf5ff")
                + tarjeta_html("Coeficiente total de X", valor_a_texto(extra['coef_total']), color="#2563eb", fondo="#eff6ff")
                + matriz_a_html(nombre_res, X, "Solución X", f"{len(X)}x{len(X[0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver procedimiento completo</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            )
            registrar_resultado(
                nombre_res, X, f"Solución de {txt_eq_izq.value.strip()} = {txt_eq_der.value.strip()}",
                proc, procedimiento_html=proc_html, resumen=resumen, tipo='ecuación-lineal'
            )
            info = obtener_info(nombre_res)
            display(HTML(html_resultado_operacion(nombre_res, info)))
            refrescar_panel_guardadas()
            refrescar_selectores()
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_verificar_id(_):
    with salida_identidad:
        clear_output()
        try:
            datos = _datos_registrados()
            ok, proc, extra = verificar_identidad_matricial(txt_id_izq.value.strip(), txt_id_der.value.strip(), datos)
            titulo = "La identidad es verdadera" if ok else "La identidad es falsa"
            color = "#16a34a" if ok else "#dc2626"
            fondo = "#f0fdf4" if ok else "#fef2f2"
            display(HTML(
                tarjeta_html(titulo,
                    f"<b>{txt_id_izq.value.strip()}</b> {'=' if ok else '≠'} <b>{txt_id_der.value.strip()}</b>",
                    color=color, fondo=fondo)
                + matriz_a_html("Lado izquierdo", extra['izquierda'], "Evaluación izquierda", f"{len(extra['izquierda'])}x{len(extra['izquierda'][0])}")
                + matriz_a_html("Lado derecho", extra['derecha'], "Evaluación derecha", f"{len(extra['derecha'])}x{len(extra['derecha'][0])}")
                + matriz_a_html("Diferencia", extra['diferencia'], "Izquierda - derecha", f"{len(extra['diferencia'])}x{len(extra['diferencia'][0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver procedimiento completo</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_resolver_param(_):
    with salida_param_det:
        clear_output()
        try:
            datos = _datos_registrados()
            soluciones, proc, extra = resolver_parametro_determinante(
                txt_expr_param.value.strip(),
                variable=txt_var_param.value.strip(),
                valor_objetivo=parsear_expresion(txt_valor_param.value.strip()),
                datos=datos
            )
            soluciones_txt = ', '.join(map(str, soluciones)) if soluciones else 'sin solución'
            display(HTML(
                tarjeta_html("Parámetro en determinante",
                    f"Soluciones para <b>{txt_var_param.value.strip()}</b>: <b>{soluciones_txt}</b>",
                    color="#d97706", fondo="#fff7ed")
                + matriz_a_html("M", extra['matriz'], f"Expresión: {txt_expr_param.value.strip()}", f"{len(extra['matriz'])}x{len(extra['matriz'][0])}")
                + tarjeta_html("Determinante obtenido", sp.sstr(extra['determinante']), color="#2563eb", fondo="#eff6ff")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver procedimiento completo</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_validar_prob(_):
    with salida_prob:
        clear_output()
        try:
            if not selector_prob_P.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return
            nombreP = selector_prob_P.value
            P = obtener_matriz(nombreP)
            ok, proc, datos = es_matriz_probabilidad(P)
            display(HTML(
                tarjeta_html(
                    "Validación de matriz de probabilidad",
                    f"La matriz <b>{nombreP}</b> {'sí' if ok else 'no'} es una matriz de probabilidad.",
                    color="#16a34a" if ok else "#dc2626",
                    fondo="#f0fdf4" if ok else "#fef2f2"
                )
                + matriz_a_html(nombreP, P, "Matriz evaluada", f"{len(P)}x{len(P[0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver verificación</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_validar_prob_prod(_):
    with salida_prob:
        clear_output()
        try:
            if not selector_prob_P.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return
            nombreP = selector_prob_P.value
            nombreQ = selector_prob_Q.value
            P = obtener_matriz(nombreP)
            Q = obtener_matriz(nombreQ)
            ok, proc, datos = verificar_probabilidad_producto(P, Q)
            display(HTML(
                tarjeta_html(
                    "Validación de producto de probabilidad",
                    f"El producto <b>{nombreP}{nombreQ}</b> {'sí' if ok else 'no'} es una matriz de probabilidad.",
                    color="#16a34a" if ok else "#dc2626",
                    fondo="#f0fdf4" if ok else "#fef2f2"
                )
                + matriz_a_html(nombreP + nombreQ, datos['producto'], "Producto evaluado", f"{len(datos['producto'])}x{len(datos['producto'][0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver verificación</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_crear_casos(_):
    with salida_casos:
        clear_output()
        try:
            nombre = txt_nombre_casos.value.strip()
            if not nombre:
                raise ValueError("Debes indicar un nombre para la matriz.")
            M, proc = crear_matriz_por_casos(
                filas_casos.value, cols_casos.value,
                txt_cond_casos.value.strip(), txt_si_casos.value.strip(), txt_no_casos.value.strip()
            )
            registrar_resultado(
                nombre, M, "Matriz creada por casos", proc,
                resumen=f"Se creó <b>{nombre}</b> por casos con condición <b>{txt_cond_casos.value.strip()}</b>.",
                tipo='crear-por-casos'
            )
            info = obtener_info(nombre)
            display(HTML(html_resultado_operacion(nombre, info)))
            refrescar_panel_guardadas()
            refrescar_selectores()
        except Exception as e:
            display(HTML(html_error(str(e))))


def accion_auto_inversa(_):
    with salida_auto_inv:
        clear_output()
        try:
            if not selector_auto_inv.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return
            nombre = selector_auto_inv.value
            A = obtener_matriz(nombre)
            ok, proc, datos = verificar_auto_inversa(A)
            display(HTML(
                tarjeta_html(
                    "Verificación de auto-inversa",
                    f"La matriz <b>{nombre}</b> {'sí' if ok else 'no'} es su propia inversa.",
                    color="#16a34a" if ok else "#dc2626",
                    fondo="#f0fdf4" if ok else "#fef2f2"
                )
                + matriz_a_html(nombre + "²", datos['A2'], "Producto A·A", f"{len(datos['A2'])}x{len(datos['A2'][0])}")
                + matriz_a_html("I", datos['I'], "Identidad del mismo orden", f"{len(datos['I'])}x{len(datos['I'][0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver procedimiento</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_resolver_poly(_):
    with salida_poly:
        clear_output()
        try:
            if not selector_poly_A.options:
                display(HTML(html_info("No hay matrices disponibles.")))
                return
            nombre = selector_poly_A.value
            A = obtener_matriz(nombre)
            soluciones, proc, extra = resolver_coeficientes_polinomio_matriz(
                A,
                variables=(txt_var_b.value.strip() or 'b', txt_var_c.value.strip() or 'c')
            )
            sol_txt = ', '.join(map(str, soluciones)) if soluciones else 'sin solución'
            display(HTML(
                tarjeta_html(
                    "Coeficientes del polinomio matricial",
                    f"Soluciones encontradas: <b>{sol_txt}</b>",
                    color="#d97706", fondo="#fff7ed"
                )
                + matriz_a_html("Expresión", extra['expresion'], "A² + v1·A + v2·I", f"{len(extra['expresion'])}x{len(extra['expresion'][0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver procedimiento</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

def accion_resolver_vars(_):
    with salida_vars:
        clear_output()
        try:
            datos = _datos_registrados()
            variables = [v.strip() for v in txt_vars_lista.value.split(',') if v.strip()]
            soluciones, proc, extra = resolver_variables_escalaras_en_igualdad(
                txt_vars_izq.value.strip(),
                txt_vars_der.value.strip(),
                variables,
                datos=datos
            )
            sol_txt = ', '.join(map(str, soluciones)) if soluciones else 'sin solución'
            display(HTML(
                tarjeta_html(
                    "Variables escalares",
                    f"Soluciones encontradas: <b>{sol_txt}</b>",
                    color="#d97706", fondo="#fff7ed"
                )
                + matriz_a_html("Lado izquierdo", extra['izquierda'], "Evaluación izquierda", f"{len(extra['izquierda'])}x{len(extra['izquierda'][0])}")
                + matriz_a_html("Lado derecho", extra['derecha'], "Evaluación derecha", f"{len(extra['derecha'])}x{len(extra['derecha'][0])}")
                + "<details><summary style='cursor:pointer; font-weight:700;'>Ver procedimiento</summary>"
                + bloque_texto_pre(proc)
                + "</details>"
            ))
        except Exception as e:
            display(HTML(html_error(str(e))))

btn_resolver_ax.on_click(accion_resolver_ax)
btn_resolver_xa.on_click(accion_resolver_xa)
btn_resolver_eq.on_click(accion_resolver_ec_lineal)
btn_verificar_id.on_click(accion_verificar_id)
btn_resolver_param.on_click(accion_resolver_param)
btn_validar_P.on_click(accion_validar_prob)
btn_validar_PQ.on_click(accion_validar_prob_prod)
btn_crear_casos.on_click(accion_crear_casos)
btn_auto_inv.on_click(accion_auto_inversa)
btn_poly.on_click(accion_resolver_poly)
btn_resolver_vars.on_click(accion_resolver_vars)

resolvedor_tabs = widgets.Tab(children=[
    widgets.VBox([
        widgets.HTML(tip_uso_html("P2.7(b), P2.10(b)", "resolver ecuaciones del tipo AX=B usando la inversa de A", "Primero crea A y B en la pestaña manual.")),
        widgets.HTML("<b>Resolver AX = B</b>"),
        widgets.HBox([selector_ax_A, selector_ax_B, selector_metodo_ax]),
        widgets.HBox([txt_nombre_ax, btn_resolver_ax]),
        salida_resolver_ax
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P2.9", "resolver ecuaciones del tipo XA=B aplicando X = BA⁻¹")),
        widgets.HTML("<b>Resolver XA = B</b>"),
        widgets.HBox([selector_xa_A, selector_xa_B, selector_metodo_xa]),
        widgets.HBox([txt_nombre_xa, btn_resolver_xa]),
        salida_resolver_xa
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P1.3(c), P1.6", "despejar X en ecuaciones como 3X-2A=B o 2X+AB-3B=-3X", "Usa nombres de matrices como A, B, C y la variable X.")),
        widgets.HTML("<b>Resolver ecuación lineal en X</b>"),
        widgets.HTML("Ejemplo: <code>3*X - 2*A</code> y <code>B</code>."),
        txt_eq_izq,
        txt_eq_der,
        widgets.HBox([txt_nombre_eq, btn_resolver_eq]),
        salida_ec_lineal
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P1.5(a), P1.5(b), P1.5(c), P1.7", "comparar dos expresiones matriciales y concluir si son iguales", "Ejemplo: <code>(A+B)*(A-B)</code> frente a <code>A**2 - B**2</code>.")),
        widgets.HTML("<b>Verificar identidad matricial</b>"),
        widgets.HTML("Ejemplo: <code>(A+B)*(A-B)</code> y <code>A**2 - B**2</code>."),
        txt_id_izq,
        txt_id_der,
        btn_verificar_id,
        salida_identidad
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P2.2, P2.3, P2.4, P2.5", "hallar parámetros como k, t, a o p a partir de un determinante", "Usa la variable en el campo Variable y el valor deseado del determinante en Objetivo.")),
        widgets.HTML("<b>Resolver parámetro en determinante</b>"),
        widgets.HTML("Ejemplo: <code>A - t*I</code> con objetivo <code>0</code>."),
        widgets.HBox([txt_expr_param, txt_var_param, txt_valor_param]),
        btn_resolver_param,
        salida_param_det
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P2.6", "comprobar si A⁻¹ = A verificando que A² = I")),
        widgets.HTML("<b>Verificar si A es su propia inversa</b>"),
        widgets.HBox([selector_auto_inv]),
        btn_auto_inv,
        salida_auto_inv
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P1.8", "hallar coeficientes escalares b y c para que se cumpla A²+bA+cI=O")),
        widgets.HTML("<b>Resolver A² + bA + cI = O</b>"),
        widgets.HBox([selector_poly_A, txt_var_b, txt_var_c]),
        btn_poly,
        salida_poly
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P1.9", "resolver variables escalares comparando dos matrices o vectores", "Puedes usar <code>col(x,y)</code>, <code>row(x,y)</code> o <code>Matrix([...])</code>.")),
        widgets.HTML("<b>Resolver variables escalares en una igualdad</b>"),
        widgets.HTML("Usa <code>col(x,y)</code>, <code>row(x,y)</code> o <code>Matrix([...])</code>."),
        widgets.HBox([txt_vars_lista]),
        txt_vars_izq,
        txt_vars_der,
        btn_resolver_vars,
        salida_vars
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P1.10(a), P1.10(b)", "verificar si una matriz, PQ o P² es de probabilidad", "El bloque revisa no negatividad y suma 1 por fila.")),
        widgets.HTML("<b>Matrices de probabilidad</b>"),
        widgets.HBox([selector_prob_P, selector_prob_Q]),
        widgets.HBox([btn_validar_P, btn_validar_PQ]),
        salida_prob
    ]),
    widgets.VBox([
        widgets.HTML(tip_uso_html("P1.6", "construir matrices definidas por condición, por ejemplo i<=j", "Escribe una condición en i y j; si se cumple usa la fórmula 'Si', si no, la fórmula 'No'.")),
        widgets.HTML("<b>Crear matriz por casos</b>"),
        widgets.HBox([txt_nombre_casos, filas_casos, cols_casos]),
        widgets.HBox([txt_cond_casos, txt_si_casos, txt_no_casos]),
        btn_crear_casos,
        salida_casos
    ]),
])
resolvedor_tabs.set_title(0, "AX = B")
resolvedor_tabs.set_title(1, "XA = B")
resolvedor_tabs.set_title(2, "Ecuación lineal")
resolvedor_tabs.set_title(3, "Identidad")
resolvedor_tabs.set_title(4, "Parámetro det")
resolvedor_tabs.set_title(5, "Auto-inversa")
resolvedor_tabs.set_title(6, "A²+bA+cI")
resolvedor_tabs.set_title(7, "Variables escalares")
resolvedor_tabs.set_title(8, "Probabilidad")
resolvedor_tabs.set_title(9, "Por casos")

tab_resolvedores = widgets.VBox([
    widgets.HTML(titulo_seccion_html("Resolvedores y pruebas", "Bloques listos para ensayar ejercicio por ejercicio de las prácticas")), 
    resolvedor_tabs
])

### Tabs

Bloque interno de la interfaz.

In [ ]:
# ---------- TABS ----------
tabs = widgets.Tab(children=[
    tab_crear_manual,
    tab_crear_formula,
    tab_transformar,
    tab_operar,
    tab_det,
    tab_resolvedores,
    tab_guardadas,
    tab_proc
])

tabs.set_title(0, "Crear manual")
tabs.set_title(1, "Por fórmula · P1.3")
tabs.set_title(2, "Transformar")
tabs.set_title(3, "Operar")
tabs.set_title(4, "Det / Inversa · P2")
tabs.set_title(5, "Resolvedores guía")
tabs.set_title(6, "Matrices guardadas")
tabs.set_title(7, "Procedimientos")

ui = widgets.VBox([
    widgets.HTML("<div style='background:linear-gradient(135deg,#0f172a,#1e3a8a); color:white; padding:16px 18px; border-radius:14px; margin-bottom:10px;'><div style='font-size:24px; font-weight:800;'>Notebook de matrices</div><div style='margin-top:6px; color:#dbeafe;'>Interfaz separada por bloques para probar cada ejercicio de las prácticas con nombres automáticos y procedimientos visibles.</div></div>"),
    tabs
], layout=widgets.Layout(width='100%'))

### Init

Bloque interno de la interfaz.

In [ ]:
# ---------- INIT ----------
generar_grid_manual(filas_manual.value, cols_manual.value)
refrescar_selectores()
actualizar_visibilidad_operacion()
display(ui)